# Fase 3.3 — Experimento de compressão e verificação de robustez

**Por que existe:** a Fase 3 ampliou a detecção para múltiplos geradores de imagens sintéticas, mas o modelo ainda apresenta dificuldade para reconhecer imagens falsas: no teste atual, o recall de falsas foi de 60,7%, enquanto a taxa de verdadeiros negativos para imagens reais foi de 81,2%. Além disso, a análise do manifesto identificou uma diferença na distribuição dos formatos originais entre as classes. Este experimento investiga se uma maior variação de compressão JPEG durante o treinamento ajuda o modelo a aprender características mais robustas, reduzindo sua dependência de possíveis artefatos de processamento.

**O que muda:** em relação ao treinamento anterior, a transformação aleatória de JPEG passa de probabilidade de 30% para 100%, com qualidade variando de 55 a 95. As demais transformações são mantidas para que o efeito dessa alteração possa ser avaliado de forma mais controlada. O experimento deve utilizar uma pasta de saída própria, preservar os checkpoints anteriores e partir dos pesos de inicialização definidos para a Fase 3, sem retomar diretamente o estado do otimizador do treinamento anterior.

**Fluxo:** configuração e conferência dos caminhos → carregamento do conjunto de dados existente → verificação da distribuição das classes, formatos e resoluções → aplicação das transformações de treino → inicialização do modelo e dos pesos → treinamento controlado → seleção do checkpoint pela validação → calibração do limiar em conjunto separado → avaliação final no teste → comparação com a Fase 3 original → validação da inferência usada pelo futuro front-end.

**Critérios de avaliação:** a comparação deve incluir acurácia geral, acurácia balanceada, AUC, precisão, recall de imagens falsas e taxa de verdadeiros negativos para imagens reais. Também devem ser observados os resultados por gerador e por origem das imagens. A meta experimental é atingir pelo menos 85% de acurácia, verificando separadamente o desempenho nas duas classes. O limiar de classificação deve ser escolhido na validação e mantido fixo durante a avaliação final no teste.

**Cuidados metodológicos:**
- O experimento deve manter os conjuntos de treino, validação e teste separados, sem utilizar o teste para escolher transformações, épocas, checkpoint ou limiar.
- A recompressão JPEG é uma hipótese experimental, não uma garantia de melhoria. Ela pode reduzir alguns atalhos de compressão, mas também pode remover sinais úteis para a classificação.
- Converter imagens para JPEG não elimina necessariamente todas as diferenças associadas ao formato original, à resolução, à origem ou ao processamento anterior.
- O notebook salvo anteriormente não correspondia integralmente à execução que produziu os resultados analisados. Os caminhos, os geradores reservados e os checkpoints devem ser confirmados antes do treinamento.
- Os checkpoints e resultados da Fase 3 original devem ser preservados para permitir comparação e recuperação.
- O pré-processamento aplicado na inferência deve corresponder ao utilizado pelo checkpoint escolhido. A integração com o front-end pode começar antes de atingir a meta, mas os resultados apresentados precisam refletir as métricas efetivamente medidas.

**Limitação declarada:** este experimento avalia uma alteração específica no aumento de dados. Uma melhora não demonstrará, por si só, que o modelo deixou de utilizar atalhos relacionados ao conjunto de dados. Caso a meta não seja atingida, os resultados por classe, gerador, formato e resolução deverão orientar o próximo experimento, em vez de realizar mudanças simultâneas e sem controle.

In [1]:
!pip install -q datasets scikit-learn

### Configuração (ajuste só aqui)

In [2]:
import os, io, csv, json, math, time, random, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFilter
from sklearn.metrics import confusion_matrix, roc_auc_score

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"

CFG = dict(
    RUN_NAME="fase3_3_compressao_v1",        # pasta do experimento (dentro de .../modelos/)
    PHASE2_CKPT="/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/phase2_final.pt",

    # --- dados (OpenFake, em streaming) ---
    HF_REPO="ComplexDataLab/OpenFake", HF_CONFIG="core",
    DATA_DIR="/content/data/p3_holdout_v2",
    SHORT_SIDE=512,                       # lado curto máximo salvo (reduz disco e iguala resolução de reais e fakes)
    JPEG_QUALITY=92,                      # TODAS as imagens salvas em JPEG com a mesma qualidade (reduz atalho de formato)
    MIN_SIDE=256,                         # descarta imagens menores que isso
    CAP_FAKE_TRAIN=100, CAP_FAKE_TEST=40, # imagens fake por gerador (treino / teste)
    PRIORITY_MULT=3,                      # geradores-alvo ganham cota multiplicada por isso
    PRIORITY_KEYS=("gpt-image", "nano-banana", "imagen", "dall"),
    CAP_REAL_TRAIN=1500, CAP_REAL_TEST=500,   # imagens reais por FONTE (laion, pexels, docci, imagenet...)
    MAX_SCAN_TRAIN=30000, MAX_SCAN_TEST=15000, MAX_MINUTES=30,   # limites de varredura do stream, por split
    HOLDOUT=("gpt-image-2", "nano-banana-pro"),     # geradores FORA do treino: medem generalização para gerador não visto
    REPLAY_DIR="/content/data/faces", REPLAY_PER_CLASS=1500,     # opcional: reaproveita faces da Fase 2 se a pasta existir

    # --- treino ---
    SEED=42, CROP=224, BATCH_SIZE=64, NUM_WORKERS=min(4, os.cpu_count() or 2),
    EPOCHS=20, WARMUP=1, PATIENCE=3,
    LR_VIT=2e-5, LR_FFT=5e-5, LR_CNN=5e-5, LR_HEAD=1e-4,   # menores que na Fase 2: é ajuste fino, não treino do zero
    WEIGHT_DECAY=1e-4, LABEL_SMOOTHING=0.1, MAX_GRAD_NORM=1.0,
    VAL_FRAC=0.10,                        # fração do pool de treino para CADA validação (early stopping e calibração)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Ative a GPU: Ambiente de execução > Alterar o tipo de ambiente de execução."
print("GPU:", torch.cuda.get_device_name(0))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed(CFG["SEED"])
torch.backends.cudnn.benchmark = True

GPU: Tesla T4


Teste Token Hugging Face

In [3]:

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if hf_token and hf_token.startswith("hf_"):
    print("Token encontrado e carregado.")
else:
    print("Token ausente ou com formato inesperado.")


Token encontrado e carregado.


### Drive, pastas e checkpoint da Fase 2

In [4]:
import shutil
from google.colab import drive
drive.mount("/content/drive")

assert os.path.isfile(CFG["PHASE2_CKPT"]), f"Não achei {CFG['PHASE2_CKPT']}: ajuste CFG['PHASE2_CKPT'] na célula de configuração."
DRIVE_ROOT = os.path.dirname(os.path.dirname(CFG["PHASE2_CKPT"]))      # .../modelos
RUN_DIR = os.path.join(DRIVE_ROOT, CFG["RUN_NAME"])
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
LAST_CKPT = os.path.join(RUN_DIR, "last.pt")        # estado completo (retomada)
BEST_CKPT = os.path.join(RUN_DIR, "best.pt")        # só os melhores pesos
HISTORY_CSV = os.path.join(RUN_DIR, "history.csv")


def free_gb(path):
    # espaço livre em GB (devolve nan se o caminho não permitir consulta)
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float("nan")


print("Pasta do experimento:", RUN_DIR)
print(f"Livre no disco local do Colab: {free_gb('/content'):.1f} GB | no Drive: {free_gb('/content/drive'):.1f} GB")
print("Estimativa de disco local para os dados: ~2 a 4 GB (com as cotas padrão).")

Mounted at /content/drive
Pasta do experimento: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1
Livre no disco local do Colab: 207.5 GB | no Drive: 10.3 GB
Estimativa de disco local para os dados: ~2 a 4 GB (com as cotas padrão).


In [5]:
import os

ROOT = "/content/drive/MyDrive/TCC_Final"

print("Arquivos de modelo e resultados encontrados:\n")

for pasta_atual, subpastas, arquivos in os.walk(ROOT):
    subpastas[:] = [
        p for p in subpastas
        if p not in {".git", "__pycache__"}
    ]

    for nome in arquivos:
        if nome.lower().endswith((
            ".pt", ".pth", ".csv", ".json", ".ipynb"
        )):
            caminho = os.path.join(pasta_atual, nome)
            print(caminho)


Arquivos de modelo e resultados encontrados:

/content/drive/MyDrive/TCC_Final/modelos/phase1_best_model.pt
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/best.pt
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/history.csv
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/last.pt
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/final_metrics.json
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/phase2_final.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/history.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/best.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/last.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/per_group.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/f

In [6]:
import os
import torch

candidatos = [
    os.path.join(ROOT, "modelos", "fase3_multigerador", "best.pt"),
    os.path.join(ROOT, "modelos", "fase3_multigerador", "last.pt"),
    os.path.join(ROOT, "modelos", "fase3_multigerador", "phase3_final.pt"),
]

for caminho in candidatos:
    if not os.path.isfile(caminho):
        print("\nNÃO ENCONTRADO:", caminho)
        continue

    ck = torch.load(
        caminho,
        map_location="cpu",
        weights_only=False
    )

    print("\nARQUIVO:", caminho)
    print("Chaves:", list(ck.keys()))

    if "cfg" in ck:
        print("Configuração:")
        print(ck["cfg"])

    if "cfg_phase3" in ck:
        print("Configuração da Fase 3:")
        print(ck["cfg_phase3"])

    if "epoch" in ck:
        print("Época:", ck["epoch"])

    if "val" in ck:
        print("Validação:", ck["val"])

    if "threshold" in ck:
        print("Limiar:", ck["threshold"])

    if "temperature" in ck:
        print("Temperatura:", ck["temperature"])



ARQUIVO: /content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/best.pt
Chaves: ['model', 'epoch', 'val', 'cfg']
Configuração:
{'RUN_NAME': 'fase3_multigerador', 'PHASE2_CKPT': '/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/phase2_final.pt', 'HF_REPO': 'ComplexDataLab/OpenFake', 'HF_CONFIG': 'core', 'DATA_DIR': '/content/data/p3', 'SHORT_SIDE': 512, 'JPEG_QUALITY': 92, 'MIN_SIDE': 256, 'CAP_FAKE_TRAIN': 100, 'CAP_FAKE_TEST': 40, 'PRIORITY_MULT': 3, 'PRIORITY_KEYS': ('gpt-image', 'nano-banana', 'imagen', 'dall'), 'CAP_REAL_TRAIN': 1500, 'CAP_REAL_TEST': 500, 'MAX_SCAN_TRAIN': 30000, 'MAX_SCAN_TEST': 15000, 'MAX_MINUTES': 30, 'HOLDOUT': ('flux.2-dev', 'sd-3.5'), 'REPLAY_DIR': '/content/data/faces', 'REPLAY_PER_CLASS': 1500, 'SEED': 42, 'CROP': 224, 'BATCH_SIZE': 64, 'NUM_WORKERS': 4, 'EPOCHS': 8, 'WARMUP': 1, 'PATIENCE': 3, 'LR_VIT': 2e-05, 'LR_FFT': 5e-05, 'LR_CNN': 5e-05, 'LR_HEAD': 0.0001, 'WEIGHT_DECAY': 0.0001, 'LABEL_SMOOTHING': 0.1, 'MAX_GRAD_NORM': 1.0, 'V

### Funções de coleta (amostragem por gerador a partir do stream)

In [7]:
def safe_name(s):
    # troca caracteres estranhos (barra, espaço) por '-': o nome do gerador vira parte do nome do arquivo
    return "".join(ch if (ch.isalnum() or ch in "-.") else "-" for ch in str(s))


def is_priority(name):
    # geradores-alvo (ChatGPT / Gemini / Imagen / DALL-E) ganham cota maior
    return any(k in str(name).lower() for k in CFG["PRIORITY_KEYS"])


def label_to_str(lab, label_names):
    # aceita o rótulo como texto ("real"/"fake") ou como inteiro, desde que haja os nomes das classes
    if isinstance(lab, (int, np.integer)):
        if not label_names:
            raise ValueError("Rótulo inteiro sem nomes de classe: não sei qual é real e qual é fake.")
        lab = label_names[int(lab)]
    return str(lab).strip().lower()


def collect_split(rows, split, cap_fake, cap_real, max_scan, label_names=None, exclude=()):
    # rows: iterável de dicts {"image": {"bytes": ...}, "label": ..., "model": ...}
    # Salva até `cap_*` imagens por grupo (fake: por gerador | real: por fonte) em DATA_DIR/<split>/<classe>/
    # e devolve o manifesto (lista de dicts com caminho, rótulo, grupo e dados do arquivo original).
    out_root = os.path.join(CFG["DATA_DIR"], split)
    for cls in ("real", "fake"):
        os.makedirs(os.path.join(out_root, cls), exist_ok=True)
    counts, manifest = {}, []
    t0, scanned = time.time(), 0
    for row in rows:
        scanned += 1
        if scanned > max_scan or (time.time() - t0) / 60 > CFG["MAX_MINUTES"]:
            break                                                    # limite de varredura ou de tempo
        lab = label_to_str(row["label"], label_names)                # levanta ValueError se o rótulo for inteiro sem nomes de classe
        name = str(row["model"])                                   # gerador (fake) ou fonte (real)
        if lab not in ("real", "fake"):
            continue
        if lab == "fake" and name in exclude:
            continue                                                 # hold-out: não entra neste split
        cap = cap_real if lab == "real" else cap_fake * (CFG["PRIORITY_MULT"] if is_priority(name) else 1)
        key = (lab, name)
        if counts.get(key, 0) >= cap:
            continue                                                 # cota deste grupo já preenchida
        try:
            raw = row["image"]
            im = Image.open(io.BytesIO(raw["bytes"])) if isinstance(raw, dict) else raw
            fmt, (w0, h0) = im.format, im.size                       # formato e tamanho ORIGINAIS (para a checagem de atalhos)
            im = im.convert("RGB")
        except Exception:
            continue                                                 # arquivo corrompido: ignora
        if min(im.size) < CFG["MIN_SIDE"]:
            continue
        s = min(im.size)
        if s > CFG["SHORT_SIDE"]:                                    # reduz só quando maior; nunca amplia
            k = CFG["SHORT_SIDE"] / s
            im = im.resize((round(im.width * k), round(im.height * k)), Image.LANCZOS)
        idx = counts.get(key, 0)
        path = os.path.join(out_root, lab, f"{safe_name(name)}__{idx:05d}.jpg")
        im.save(path, format="JPEG", quality=CFG["JPEG_QUALITY"])
        counts[key] = idx + 1
        manifest.append({"path": path, "label": 0 if lab == "real" else 1, "group": name,
                         "orig_format": fmt, "orig_w": w0, "orig_h": h0})
        if scanned % 2000 == 0:
            print(f"  [{split}] varridas={scanned} | salvas={len(manifest)} | {(time.time() - t0) / 60:.1f} min")
    n_real = sum(1 for m in manifest if m["label"] == 0)
    print(f"[{split}] varridas={scanned} | salvas={len(manifest)} (reais={n_real}, fakes={len(manifest) - n_real}) "
          f"| grupos fake={len({m['group'] for m in manifest if m['label'] == 1})} "
          f"| reais por fonte={len({m['group'] for m in manifest if m['label'] == 0})} | {(time.time() - t0) / 60:.1f} min")
    return manifest

### Download amostrado do OpenFake (streaming: nada é baixado por inteiro)

In [8]:
manifest_path = os.path.join(CFG["DATA_DIR"], "manifest.json")

if os.path.isfile(manifest_path):
    MANIFEST = json.load(open(manifest_path))
    print("Manifesto existente reaproveitado:", manifest_path)
else:
    from datasets import load_dataset, Image as HFImage
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")        # opcional (Secrets do Colab); sem token o acesso anônimo também funciona
    except Exception:
        hf_token = None

    def open_stream(hf_split):
        # abre o split em streaming, sem decodificar imagens (só bytes) e com embaralhamento por shard + buffer pequeno
        ds = load_dataset(CFG["HF_REPO"], CFG["HF_CONFIG"], split=hf_split, streaming=True, token=hf_token)
        ds = ds.cast_column("image", HFImage(decode=False))
        feats = ds.features or {}
        names = getattr(feats.get("label"), "names", None)       # nomes de classe, se o rótulo for inteiro
        return ds.shuffle(seed=CFG["SEED"], buffer_size=300), names   # buffer pequeno: cada linha tem ~1 MB

    MANIFEST = {}
    for hf_split in ("train", "validation"):       # pool de treino: usa "train"; se o split não abrir, cai para "validation"
        try:
            ds, names = open_stream(hf_split)
            MANIFEST["train"] = collect_split(ds, "train", CFG["CAP_FAKE_TRAIN"], CFG["CAP_REAL_TRAIN"],
                                              CFG["MAX_SCAN_TRAIN"], names, exclude=CFG["HOLDOUT"])
            break
        except Exception as e:
            print(f"Falha ao ler o split '{hf_split}': {type(e).__name__}: {e}")
    assert "train" in MANIFEST, "Não consegui ler nenhum split de treino do OpenFake (veja o erro acima)."

    ds, names = open_stream("test")                # teste: split separado, com TODOS os geradores (inclusive hold-out)
    MANIFEST["test"] = collect_split(ds, "test", CFG["CAP_FAKE_TEST"], CFG["CAP_REAL_TEST"],
                                     CFG["MAX_SCAN_TEST"], names, exclude=())
    json.dump(MANIFEST, open(manifest_path, "w"))


train_fake = {
    r["group"]
    for r in MANIFEST["train"]
    if r["label"] == 1
}

test_fake = {
    r["group"]
    for r in MANIFEST["test"]
    if r["label"] == 1
}

holdout = set(CFG["HOLDOUT"])

presentes_no_teste = holdout & test_fake
ausentes_no_teste = holdout - test_fake
presentes_no_treino = holdout & train_fake

print("Geradores reservados:", sorted(holdout))
print("Geradores reservados presentes no teste:", sorted(presentes_no_teste))
print("Geradores reservados ausentes do teste:", sorted(ausentes_no_teste))
print("Geradores reservados encontrados no treino:", sorted(presentes_no_treino))

if ausentes_no_teste:
    raise ValueError(
        "Experimento interrompido: pelo menos um gerador reservado "
        "não apareceu no teste. Confira os nomes exatos e a coleta."
    )

if presentes_no_treino:
    raise ValueError(
        "Experimento interrompido: um gerador reservado apareceu "
        "no conjunto de treino."
    )

print("OK: os geradores reservados estão no teste e não aparecem no treino.")

print("Geradores fake no teste:", sorted(test_fake))
print(
    "Fontes reais no teste:",
    sorted({
        r["group"]
        for r in MANIFEST["test"]
        if r["label"] == 0
    })
)

alvo = sorted(g for g in test_fake if is_priority(g))
print("Geradores-alvo presentes:", alvo or "Nenhum")


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/608 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/608 [00:00<?, ?it/s]

  [train] varridas=2000 | salvas=1856 | 4.6 min


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  [train] varridas=4000 | salvas=3428 | 9.1 min
  [train] varridas=30000 | salvas=7983 | 25.6 min
[train] varridas=30001 | salvas=7983 (reais=3000, fakes=4983) | grupos fake=74 | reais por fonte=2 | 25.6 min


Resolving data files:   0%|          | 0/608 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/608 [00:00<?, ?it/s]

[test] varridas=15001 | salvas=1947 (reais=1000, fakes=947) | grupos fake=20 | reais por fonte=2 | 3.8 min
Geradores reservados: ['gpt-image-2', 'nano-banana-pro']
Geradores reservados presentes no teste: ['gpt-image-2', 'nano-banana-pro']
Geradores reservados ausentes do teste: []
Geradores reservados encontrados no treino: []
OK: os geradores reservados estão no teste e não aparecem no treino.
Geradores fake no teste: ['aurora-20-1-25', 'ernie-image', 'ernie-image-turbo', 'flux.2-klein-9b', 'frames-23-1-25', 'gpt-image-1.5', 'gpt-image-2', 'halfmoon-4-4-25', 'ideogram-2.0', 'illustrious', 'lumina-17-2-25', 'midjourney-7', 'nano-banana-pro', 'recraft-v2', 'recraft-v3', 'seedream-v5.0', 'sora-2', 'veo-3', 'wan-video-2.5', 'z-image-turbo']
Fontes reais no teste: ['docci', 'imagenet']
Geradores-alvo presentes: ['gpt-image-1.5', 'gpt-image-2', 'nano-banana-pro']


apagar daqui


In [20]:
from datetime import datetime

paths_to_check = [
    manifest_path,
    os.path.join(RUN_DIR, "final_metrics.json"),
    os.path.join(RUN_DIR, "per_group.csv"),
    os.path.join(RUN_DIR, "test_predictions.csv"),
    os.path.join(RUN_DIR, "history.csv"),
    os.path.join(RUN_DIR, "phase3_final.pt"),
]

for path in paths_to_check:
    if os.path.isfile(path):
        stat = os.stat(path)
        print(
            f"{os.path.basename(path):25s} | "
            f"{stat.st_size / 1024:.1f} KB | "
            f"modificado: {datetime.fromtimestamp(stat.st_mtime)}"
        )
    else:
        print("AUSENTE:", path)

manifest.json             | 1591.5 KB | modificado: 2026-10-22 13:12:18.765685
final_metrics.json        | 2.0 KB | modificado: 2026-10-22 13:25:36
per_group.csv             | 1.6 KB | modificado: 2026-10-22 13:25:36
test_predictions.csv      | 159.0 KB | modificado: 2026-10-22 13:25:36
history.csv               | 2.5 KB | modificado: 2026-10-22 13:25:06
phase3_final.pt           | 39675.6 KB | modificado: 2026-10-22 13:25:36


In [22]:
import os
import json
from collections import Counter

manifest_path = os.path.join(CFG["DATA_DIR"], "manifest.json")

print("Manifesto:", manifest_path)
print("Existe:", os.path.isfile(manifest_path))

if os.path.isfile(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest_audit = json.load(f)

    for split in ("train", "test"):
        rows = manifest_audit.get(split, [])
        labels = Counter(r.get("label") for r in rows)
        groups = Counter(r.get("group") for r in rows)

        print(f"\n--- {split.upper()} ---")
        print("Total:", len(rows))
        print("Rótulos:", dict(labels))
        print("Grupos distintos:", len(groups))
        print("10 grupos mais frequentes:", groups.most_common(10))

Manifesto: /content/data/p3_holdout_v2/manifest.json
Existe: True

--- TRAIN ---
Total: 7983
Rótulos: {0: 3000, 1: 4983}
Grupos distintos: 76
10 grupos mais frequentes: [('pexels', 1500), ('laion', 1500), ('gpt-image-1', 300), ('dalle-3', 300), ('imagen-4.0', 189), ('sd-1.5-epicdream', 100), ('sd-2.1', 100), ('flux.1-dev', 100), ('sdxl-touchofrealism', 100), ('sdxl-realvis-v5', 100)]

--- TEST ---
Total: 1947
Rótulos: {0: 1000, 1: 947}
Grupos distintos: 22
10 grupos mais frequentes: [('docci', 500), ('imagenet', 500), ('gpt-image-1.5', 120), ('gpt-image-2', 92), ('nano-banana-pro', 63), ('veo-3', 40), ('z-image-turbo', 40), ('ernie-image-turbo', 40), ('flux.2-klein-9b', 40), ('illustrious', 40)]


In [23]:
import shutil

backup_path = os.path.join(
    RUN_DIR,
    "manifest_audit_copy.json"
)

shutil.copy2(manifest_path, backup_path)

print("Cópia de segurança criada:", backup_path)

Cópia de segurança criada: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/manifest_audit_copy.json


In [24]:
from collections import Counter

train_rows = manifest_audit.get("train", [])
test_rows = manifest_audit.get("test", [])

def get_path(row):
    return row.get("path")

train_paths = [get_path(r) for r in train_rows]
test_paths = [get_path(r) for r in test_rows]

print("Caminhos repetidos no treino:",
      sum(n - 1 for n in Counter(train_paths).values() if n > 1))

print("Caminhos repetidos no teste:",
      sum(n - 1 for n in Counter(test_paths).values() if n > 1))

overlap = set(train_paths) & set(test_paths)
print("Caminhos em comum entre treino e teste:", len(overlap))

if overlap:
    print("Exemplos:", list(overlap)[:10])

Caminhos repetidos no treino: 0
Caminhos repetidos no teste: 0
Caminhos em comum entre treino e teste: 0


In [25]:
holdout = {"gpt-image-2", "nano-banana-pro"}

def group_name(row):
    return str(row.get("group", "")).strip().lower()

train_groups = Counter(group_name(r) for r in train_rows)
test_groups = Counter(group_name(r) for r in test_rows)

print("Geradores reservados no treino:")
for g in sorted(holdout):
    print(f"{g}: {train_groups.get(g, 0)}")

print("\nGeradores reservados no teste:")
for g in sorted(holdout):
    print(f"{g}: {test_groups.get(g, 0)}")

Geradores reservados no treino:
gpt-image-2: 0
nano-banana-pro: 0

Geradores reservados no teste:
gpt-image-2: 92
nano-banana-pro: 63


In [26]:
from collections import Counter

for nome, rows in [("TREINO", train_rows), ("TESTE", test_rows)]:
    total = len(rows)
    existentes = sum(
        1 for r in rows
        if r.get("path") and os.path.isfile(r["path"])
    )

    print(
        f"{nome}: {existentes}/{total} arquivos encontrados "
        f"({existentes / total:.1%} se total > 0 else 0)"
    )

TREINO: 7983/7983 arquivos encontrados (100.0% se total > 0 else 0)
TESTE: 1947/1947 arquivos encontrados (100.0% se total > 0 else 0)


In [27]:
import os
import json
import pandas as pd

metrics_path = os.path.join(RUN_DIR, "final_metrics.json")
group_path = os.path.join(RUN_DIR, "per_group.csv")
pred_path = os.path.join(RUN_DIR, "test_predictions.csv")
history_path = os.path.join(RUN_DIR, "history.csv")

with open(metrics_path, "r", encoding="utf-8") as f:
    final_metrics = json.load(f)

per_group = pd.read_csv(group_path)
predictions = pd.read_csv(pred_path)
history = pd.read_csv(history_path)

print("Chaves de final_metrics.json:")
print(list(final_metrics.keys()))

print("\nPredições:", predictions.shape)
print("Colunas das predições:", predictions.columns.tolist())

print("\nMétricas por grupo:", per_group.shape)
print("Colunas por grupo:", per_group.columns.tolist())

print("\nHistórico:", history.shape)
print("Colunas do histórico:", history.columns.tolist())

display(predictions.head())
display(per_group.head())
display(history.tail())

Chaves de final_metrics.json:
['temperature', 'threshold', 'auc', 'auc_ci95', 'test_0.5', 'test_thr', 'baseline_phase2', 'best_epoch', 'cfg']

Predições: (1947, 5)
Colunas das predições: ['path', 'label', 'group', 'prob_fake', 'pred_thr']

Métricas por grupo: (22, 7)
Colunas por grupo: ['classe', 'grupo', 'n', 'acerto', 'ic_lo', 'ic_hi', 'holdout']

Histórico: (18, 8)
Colunas do histórico: ['epoch', 'lr_head', 'train_loss', 'val_loss', 'val_auc', 'val_bal_acc', 'val_f1', 'sec']


,path,label,group,prob_fake,pred_thr
0,/content/data/p3_holdout_v2/test/real/docci__0...,0,docci,0.829147,1
1,/content/data/p3_holdout_v2/test/real/docci__0...,0,docci,0.554508,1
2,/content/data/p3_holdout_v2/test/real/imagenet...,0,imagenet,0.268772,0
3,/content/data/p3_holdout_v2/test/fake/veo-3__0...,1,veo-3,0.092955,0
4,/content/data/p3_holdout_v2/test/real/docci__0...,0,docci,0.105236,0


,classe,grupo,n,acerto,ic_lo,ic_hi,holdout
0,fake,flux.2-klein-9b,40,0.225,0.123160,0.375034,False
1,fake,frames-23-1-25,40,0.250,0.141870,0.401943,False
2,fake,recraft-v3,40,0.300,0.180747,0.454303,False
3,fake,aurora-20-1-25,40,0.300,0.180747,0.454303,False
4,fake,sora-2,40,0.425,0.285092,0.578052,False


,epoch,lr_head,train_loss,val_loss,val_auc,val_bal_acc,val_f1,sec
13,14,0.000030,0.426249,0.435607,0.873948,0.778480,0.834711,35.991767
14,15,0.000023,0.432953,0.431940,0.874357,0.798616,0.837953,35.940323
15,16,0.000016,0.432206,0.433502,0.873305,0.794476,0.838437,36.096170
16,17,0.000011,0.428417,0.437784,0.871279,0.786813,0.839045,35.817263
17,18,0.000006,0.434365,0.433869,0.873005,0.800954,0.843882,35.879449


In [29]:
print(predictions.columns.tolist())
print(predictions.head(3).to_string())

['path', 'label', 'group', 'prob_fake', 'pred_thr']
                                                        path  label     group  prob_fake  pred_thr
0     /content/data/p3_holdout_v2/test/real/docci__00000.jpg      0     docci   0.829147         1
1     /content/data/p3_holdout_v2/test/real/docci__00001.jpg      0     docci   0.554508         1
2  /content/data/p3_holdout_v2/test/real/imagenet__00000.jpg      0  imagenet   0.268772         0


In [30]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    balanced_accuracy_score
)

THRESHOLD = 0.470

# Valores reais e probabilidades previstas
y_true = predictions["label"].astype(int).to_numpy()
p_fake = predictions["prob_fake"].astype(float).to_numpy()

# Recalcular as classes usando o limiar definido
y_pred = (p_fake >= THRESHOLD).astype(int)

# Matriz de confusão: TN, FP, FN, TP
tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("========== AUDITORIA DAS MÉTRICAS ==========")
print(f"Total de amostras: {len(y_true)}")
print(f"Limiar: {THRESHOLD:.3f}")
print(f"AUC: {roc_auc_score(y_true, p_fake):.4f}")
print(f"Acurácia: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precisão (fake): {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall (fake): {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1-score (fake): {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Especificidade (real): {specificity:.4f}")
print(f"Acurácia balanceada: {balanced_accuracy_score(y_true, y_pred):.4f}")

print("\nMatriz de confusão:")
print(f"TN={tn} | FP={fp}")
print(f"FN={fn} | TP={tp}")

# Conferir as previsões salvas no CSV contra o limiar recalculado
if "pred_thr" in predictions.columns:
    pred_saved = predictions["pred_thr"].astype(int).to_numpy()
    differences = np.sum(pred_saved != y_pred)
    print(f"\nDivergências em pred_thr: {differences}")

    if differences == 0:
        print("OK: pred_thr corresponde ao limiar aplicado.")
    else:
        print("ATENÇÃO: pred_thr difere das classes recalculadas.")

========== AUDITORIA DAS MÉTRICAS ==========
Total de amostras: 1947
Limiar: 0.470
AUC: 0.8041
Acurácia: 0.7114
Precisão (fake): 0.7762
Recall (fake): 0.5713
F1-score (fake): 0.6582
Especificidade (real): 0.8440
Acurácia balanceada: 0.7076

Matriz de confusão:
TN=844 | FP=156
FN=406 | TP=541

Divergências em pred_thr: 0
OK: pred_thr corresponde ao limiar aplicado.


In [31]:
print(per_group.to_string(index=False))

classe             grupo   n   acerto    ic_lo    ic_hi  holdout
  fake   flux.2-klein-9b  40 0.225000 0.123160 0.375034    False
  fake    frames-23-1-25  40 0.250000 0.141870 0.401943    False
  fake        recraft-v3  40 0.300000 0.180747 0.454303    False
  fake    aurora-20-1-25  40 0.300000 0.180747 0.454303    False
  fake            sora-2  40 0.425000 0.285092 0.578052    False
  fake             veo-3  40 0.475000 0.329352 0.625029    False
  fake   halfmoon-4-4-25  35 0.485714 0.329940 0.644315    False
  fake     gpt-image-1.5 120 0.491667 0.403874 0.579976    False
  fake      midjourney-7  40 0.500000 0.351993 0.648007    False
  fake      ideogram-2.0  40 0.525000 0.374971 0.670648    False
  fake       gpt-image-2  92 0.565217 0.463321 0.661886     True
  fake        recraft-v2  37 0.594595 0.434857 0.736537    False
  fake   nano-banana-pro  63 0.619048 0.495584 0.728827     True
  fake ernie-image-turbo  40 0.625000 0.470322 0.757772    False
  fake       ernie-image 

In [32]:
print(history.to_string(index=False))

 epoch  lr_head  train_loss  val_loss  val_auc  val_bal_acc   val_f1       sec
     1 0.000100    0.761587  0.619084 0.711495     0.660031 0.680702 53.134542
     2 0.000100    0.601996  0.549128 0.789976     0.703008 0.741071 34.420882
     3 0.000099    0.560486  0.521555 0.811111     0.711363 0.774262 36.573811
     4 0.000097    0.527558  0.492948 0.834693     0.747841 0.794844 35.701399
     5 0.000094    0.513006  0.488160 0.834668     0.765660 0.816456 35.839577
     6 0.000089    0.504111  0.469737 0.850744     0.772086 0.816239 36.058820
     7 0.000084    0.486033  0.457675 0.859179     0.779665 0.828601 35.752317
     8 0.000077    0.474339  0.452339 0.861859     0.786331 0.832101 36.272643
     9 0.000070    0.464263  0.444044 0.866813     0.793428 0.837209 35.632334
    10 0.000062    0.454860  0.443665 0.867645     0.778480 0.834711 36.075720
    11 0.000054    0.449293  0.441339 0.868809     0.777998 0.827731 36.004184
    12 0.000046    0.439754  0.445001 0.866775     0

In [33]:
import os
import pandas as pd
from PIL import Image
from collections import Counter

EXTERNAL_DIR = (
    "/content/drive/MyDrive/TCC_Final/"
    "validacao_fase3/teste_externo"
)
IMAGES_DIR = os.path.join(EXTERNAL_DIR, "imagens")
LABELS_CSV = os.path.join(EXTERNAL_DIR, "rotulos.csv")

labels_df = pd.read_csv(LABELS_CSV)

required = {"arquivo", "label_real", "origem", "observacao"}
assert required.issubset(labels_df.columns), (
    f"Colunas ausentes: {required - set(labels_df.columns)}"
)

assert labels_df["arquivo"].is_unique, "Há nomes duplicados no CSV."
assert set(labels_df["label_real"].dropna().unique()).issubset({0, 1})

labels_df["caminho"] = labels_df["arquivo"].apply(
    lambda name: os.path.join(IMAGES_DIR, name)
)

labels_df["existe"] = labels_df["caminho"].apply(os.path.isfile)

print("Total no CSV:", len(labels_df))
print("Imagens encontradas:", int(labels_df["existe"].sum()))
print("Rótulos:", labels_df["label_real"].value_counts().to_dict())

display(labels_df.loc[~labels_df["existe"]])

Total no CSV: 20
Imagens encontradas: 0
Rótulos: {1: 12, 0: 8}


,arquivo,label_real,origem,observacao,caminho,existe
0,Copy of imagem_falsa (1).jpeg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
1,Copy of imagem_falsa (1).jpg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
2,Copy of imagem_falsa (2).jpeg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
3,Copy of imagem_falsa (2).jpg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
4,Copy of imagem_falsa (3).jpeg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
5,Copy of imagem_falsa (3).jpg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
6,Copy of imagem_falsa (4).jpeg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
7,Copy of imagem_falsa (4).jpg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
8,Copy of imagem_falsa (5).jpeg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False
9,Copy of imagem_falsa (5).jpg,1,desconhecido,desconhecido,/content/drive/MyDrive/TCC_Final/validacao_fas...,False


In [34]:
import hashlib
from collections import defaultdict

hash_groups = defaultdict(list)

for path in labels_df.loc[labels_df["existe"], "caminho"]:
    with open(path, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    hash_groups[digest].append(os.path.basename(path))

print("Grupos de arquivos binariamente idênticos:")
for files in hash_groups.values():
    if len(files) > 1:
        print(files)

Grupos de arquivos binariamente idênticos:


In [37]:
import os
import glob

print("EXTERNAL_DIR:", EXTERNAL_DIR)
print("Pasta existe:", os.path.isdir(EXTERNAL_DIR))

if os.path.isdir(EXTERNAL_DIR):
    print("\nArquivos na pasta:")
    for nome in os.listdir(EXTERNAL_DIR):
        print(nome)

# Procurar CSVs relacionados ao teste externo dentro do TCC_Final
SEARCH_ROOT = "/content/drive/MyDrive/TCC_Final"

matches = glob.glob(
    os.path.join(SEARCH_ROOT, "**", "*extern*csv"),
    recursive=True
)

print("\nCSV encontrados com 'extern' no nome:")
for caminho in matches:
    print(caminho)

# Procurar também arquivos de predição com outros nomes
matches_pred = glob.glob(
    os.path.join(SEARCH_ROOT, "**", "*pred*csv"),
    recursive=True
)

print("\nCSV encontrados com 'pred' no nome:")
for caminho in matches_pred:
    print(caminho)

EXTERNAL_DIR: /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo
Pasta existe: True

Arquivos na pasta:
rotulos.csv
Copy of imagem_real6.jpg
Copy of imagem_falsa (7).jpg
Copy of imagem_falsa (2).jpg
Copy of imagem_falsa (5).jpg
Copy of imagem_falsa (4).jpg
Copy of imagem_falsa (3).jpg
Copy of imagem_real2.jpg
Copy of imagem_falsa (3).jpeg
Copy of imagem_falsa (2).jpeg
Copy of imagem_real4.jpg
Copy of imagem_falsa (1).jpg
Copy of imagem_real1.jpg
Copy of imagem_falsa (4).jpeg
Copy of imagem_falsa (5).jpeg
Copy of imagem_real3.jpg
Copy of imagem_falsa (6).jpg
Copy of imagem_real8.jpg
Copy of imagem_real7.jpg
Copy of imagem_falsa (1).jpeg
Copy of imagem_real5.jpg

CSV encontrados com 'extern' no nome:
/content/drive/MyDrive/TCC_Final/teste_externo/external_predictions.csv

CSV encontrados com 'pred' no nome:
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/test_predictions.csv
/cont

In [43]:
PRED_CSV = "/content/drive/MyDrive/TCC_Final/teste_externo/external_predictions.csv"
pred_df = pd.read_csv(PRED_CSV)

print("Colunas:", pred_df.columns.tolist())
print(pred_df.head())

Colunas: ['arquivo', 'classe_real', 'prob_fake', 'pred_0.50', 'pred_limiar']
                                       arquivo classe_real  prob_fake  \
0                             techtudo-21-.jpg        FAKE   0.000550   
1                              unnamed-37-.jpg        FAKE   0.000549   
2  gemini-generated-image-6gfhpd6gfhpd6gfh.jpg        FAKE   0.000546   
3               wooden-bridge-beach-sunset.jpg        REAL   0.000545   
4   WhatsApp Image 2026-10-07 at 11.43.08.jpeg        FAKE   0.000533   

  pred_0.50 pred_limiar  
0      REAL        REAL  
1      REAL        REAL  
2      REAL        REAL  
3      REAL        REAL  
4      REAL        REAL  


In [44]:
import os
import glob
import pandas as pd

SEARCH_ROOT = "/content/drive/MyDrive/TCC_Final"

print("EXTERNAL_DIR =", EXTERNAL_DIR)
print("Pasta existe:", os.path.isdir(EXTERNAL_DIR))

# Procurar arquivos CSV com possíveis nomes de predição
patterns = [
    "**/external_predictions.csv",
    "**/predicoes_externas.csv",
    "**/*prediction*.csv",
    "**/*predicao*.csv",
    "**/*pred*.csv"
]

found = set()

for pattern in patterns:
    found.update(
        glob.glob(os.path.join(SEARCH_ROOT, pattern), recursive=True)
    )

print("\nArquivos CSV de predição encontrados:")

if found:
    for path in sorted(found):
        print(path)
else:
    print("Nenhum CSV de predição encontrado.")

# Listar os arquivos da pasta de validação externa
print("\nConteúdo de EXTERNAL_DIR:")
if os.path.isdir(EXTERNAL_DIR):
    for name in os.listdir(EXTERNAL_DIR):
        print(name)

EXTERNAL_DIR = /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo
Pasta existe: True

Arquivos CSV de predição encontrados:
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_holdout_v2/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/test_predictions.csv
/content/drive/MyDrive/TCC_Final/teste_externo/external_predictions.csv

Conteúdo de EXTERNAL_DIR:
rotulos.csv
Copy of imagem_real6.jpg
Copy of imagem_falsa (7).jpg
Copy of imagem_falsa (2).jpg
Copy of imagem_falsa (5).jpg
Copy of imagem_falsa (4).jpg
Copy of imagem_falsa (3).jpg
Copy of imagem_real2.jpg
Copy of imagem_falsa (3).jpeg
Copy of imagem_falsa (2).jpeg
Copy of imagem_real4.jpg
Copy of imagem_falsa (1).jpg
Copy of imagem_real1.jpg
Copy of imagem_falsa (4).jpeg
Copy of imagem_falsa (5).jpeg
Copy of imagem_real3.jpg
Copy 

In [46]:
import os
import glob
import pandas as pd

SEARCH_ROOT = "/content/drive/MyDrive/TCC_Final"

print("EXTERNAL_DIR =", EXTERNAL_DIR)
print("Pasta existe:", os.path.isdir(EXTERNAL_DIR))

# Procurar arquivos CSV com possíveis nomes de predição
patterns = [
    "**/external_predictions.csv",
    "**/predicoes_externas.csv",
    "**/*prediction*.csv",
    "**/*predicao*.csv",
    "**/*pred*.csv"
]

found = set()

for pattern in patterns:
    found.update(
        glob.glob(os.path.join(SEARCH_ROOT, pattern), recursive=True)
    )

print("\nArquivos CSV de predição encontrados:")

if found:
    for path in sorted(found):
        print(path)
else:
    print("Nenhum CSV de predição encontrado.")

# Listar os arquivos da pasta de validação externa
print("\nConteúdo de EXTERNAL_DIR:")
if os.path.isdir(EXTERNAL_DIR):
    for name in os.listdir(EXTERNAL_DIR):
        print(name)

EXTERNAL_DIR = /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo
Pasta existe: True

Arquivos CSV de predição encontrados:
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_holdout_v2/test_predictions.csv
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/test_predictions.csv
/content/drive/MyDrive/TCC_Final/teste_externo/external_predictions.csv

Conteúdo de EXTERNAL_DIR:
rotulos.csv
Copy of imagem_real6.jpg
Copy of imagem_falsa (7).jpg
Copy of imagem_falsa (2).jpg
Copy of imagem_falsa (5).jpg
Copy of imagem_falsa (4).jpg
Copy of imagem_falsa (3).jpg
Copy of imagem_real2.jpg
Copy of imagem_falsa (3).jpeg
Copy of imagem_falsa (2).jpeg
Copy of imagem_real4.jpg
Copy of imagem_falsa (1).jpg
Copy of imagem_real1.jpg
Copy of imagem_falsa (4).jpeg
Copy of imagem_falsa (5).jpeg
Copy of imagem_real3.jpg
Copy 

In [47]:
import os

EXTERNAL_DIR = "/content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo"

print("Pasta existe:", os.path.isdir(EXTERNAL_DIR))

if os.path.isdir(EXTERNAL_DIR):
    print("\nArquivos encontrados:")
    for nome in os.listdir(EXTERNAL_DIR):
        caminho = os.path.join(EXTERNAL_DIR, nome)
        tipo = "PASTA" if os.path.isdir(caminho) else "ARQUIVO"
        print(f"{tipo}: {nome}")

Pasta existe: True

Arquivos encontrados:
ARQUIVO: rotulos.csv
ARQUIVO: Copy of imagem_real6.jpg
ARQUIVO: Copy of imagem_falsa (7).jpg
ARQUIVO: Copy of imagem_falsa (2).jpg
ARQUIVO: Copy of imagem_falsa (5).jpg
ARQUIVO: Copy of imagem_falsa (4).jpg
ARQUIVO: Copy of imagem_falsa (3).jpg
ARQUIVO: Copy of imagem_real2.jpg
ARQUIVO: Copy of imagem_falsa (3).jpeg
ARQUIVO: Copy of imagem_falsa (2).jpeg
ARQUIVO: Copy of imagem_real4.jpg
ARQUIVO: Copy of imagem_falsa (1).jpg
ARQUIVO: Copy of imagem_real1.jpg
ARQUIVO: Copy of imagem_falsa (4).jpeg
ARQUIVO: Copy of imagem_falsa (5).jpeg
ARQUIVO: Copy of imagem_real3.jpg
ARQUIVO: Copy of imagem_falsa (6).jpg
ARQUIVO: Copy of imagem_real8.jpg
ARQUIVO: Copy of imagem_real7.jpg
ARQUIVO: Copy of imagem_falsa (1).jpeg
ARQUIVO: Copy of imagem_real5.jpg


In [53]:

import os
import pandas as pd
from pathlib import Path

EXTERNAL_DIR = (
    "/content/drive/MyDrive/TCC_Final/"
    "validacao_fase3/teste_externo"
)
LABELS_CSV = os.path.join(EXTERNAL_DIR, "rotulos.csv")
PRED_CSV = (
    "/content/drive/MyDrive/TCC_Final/"
    "teste_externo/external_predictions.csv"
)

labels_df = pd.read_csv(LABELS_CSV)
pred_df = pd.read_csv(PRED_CSV)

print("Rótulos:", len(labels_df))
print("Predições:", len(pred_df))
print("Colunas dos rótulos:", labels_df.columns.tolist())
print("Colunas das predições:", pred_df.columns.tolist())

def normalize_name(value):
    return Path(str(value).strip()).name.casefold()

labels_df["nome_normalizado"] = labels_df["arquivo"].map(normalize_name)
pred_df["nome_normalizado"] = pred_df["arquivo"].map(normalize_name)

print("\n========== NOMES EM rotulos.csv ==========")
for name in labels_df["arquivo"]:
    print(repr(name))

print("\n========== NOMES EM external_predictions.csv ==========")
for name in pred_df["arquivo"]:
    print(repr(name))

print("\n========== CORRESPONDÊNCIAS EXATAS ==========")
matched = labels_df["nome_normalizado"].isin(
    set(pred_df["nome_normalizado"])
)
print(f"Correspondências: {matched.sum()} de {len(labels_df)}")

print("\nImagens sem correspondência:")
display(
    labels_df.loc[
        ~matched, ["arquivo", "label_real", "origem", "observacao"]
    ]
)

print("\nPrimeiras linhas das predições:")
display(pred_df.head(25))


Rótulos: 20
Predições: 20
Colunas dos rótulos: ['arquivo', 'label_real', 'origem', 'observacao']
Colunas das predições: ['arquivo', 'classe_real', 'prob_fake', 'pred_0.50', 'pred_limiar']

========== NOMES EM rotulos.csv ==========
'Copy of imagem_falsa (1).jpeg'
'Copy of imagem_falsa (1).jpg'
'Copy of imagem_falsa (2).jpeg'
'Copy of imagem_falsa (2).jpg'
'Copy of imagem_falsa (3).jpeg'
'Copy of imagem_falsa (3).jpg'
'Copy of imagem_falsa (4).jpeg'
'Copy of imagem_falsa (4).jpg'
'Copy of imagem_falsa (5).jpeg'
'Copy of imagem_falsa (5).jpg'
'Copy of imagem_falsa (6).jpg'
'Copy of imagem_falsa (7).jpg'
'Copy of imagem_real1.jpg'
'Copy of imagem_real2.jpg'
'Copy of imagem_real3.jpg'
'Copy of imagem_real4.jpg'
'Copy of imagem_real5.jpg'
'Copy of imagem_real6.jpg'
'Copy of imagem_real7.jpg'
'Copy of imagem_real8.jpg'

========== NOMES EM external_predictions.csv ==========
'techtudo-21-.jpg'
'unnamed-37-.jpg'
'gemini-generated-image-6gfhpd6gfhpd6gfh.jpg'
'wooden-bridge-beach-sunset.jpg'
'W

,arquivo,label_real,origem,observacao
0,Copy of imagem_falsa (1).jpeg,1,desconhecido,desconhecido
1,Copy of imagem_falsa (1).jpg,1,desconhecido,desconhecido
2,Copy of imagem_falsa (2).jpeg,1,desconhecido,desconhecido
3,Copy of imagem_falsa (2).jpg,1,desconhecido,desconhecido
4,Copy of imagem_falsa (3).jpeg,1,desconhecido,desconhecido
5,Copy of imagem_falsa (3).jpg,1,desconhecido,desconhecido
6,Copy of imagem_falsa (4).jpeg,1,desconhecido,desconhecido
7,Copy of imagem_falsa (4).jpg,1,desconhecido,desconhecido
8,Copy of imagem_falsa (5).jpeg,1,desconhecido,desconhecido
9,Copy of imagem_falsa (5).jpg,1,desconhecido,desconhecido



Primeiras linhas das predições:


,arquivo,classe_real,prob_fake,pred_0.50,pred_limiar,nome_normalizado
0,techtudo-21-.jpg,FAKE,0.000550,REAL,REAL,techtudo-21-.jpg
1,unnamed-37-.jpg,FAKE,0.000549,REAL,REAL,unnamed-37-.jpg
2,gemini-generated-image-6gfhpd6gfhpd6gfh.jpg,FAKE,0.000546,REAL,REAL,gemini-generated-image-6gfhpd6gfhpd6gfh.jpg
3,wooden-bridge-beach-sunset.jpg,REAL,0.000545,REAL,REAL,wooden-bridge-beach-sunset.jpg
4,WhatsApp Image 2026-10-07 at 11.43.08.jpeg,FAKE,0.000533,REAL,REAL,whatsapp image 2026-10-07 at 11.43.08.jpeg
5,WhatsApp Image 2026-10-07 at 11.43.11.jpeg,REAL,0.000531,REAL,REAL,whatsapp image 2026-10-07 at 11.43.11.jpeg
6,unnamed-29-.jpg,FAKE,0.000527,REAL,REAL,unnamed-29-.jpg
7,WhatsApp Image 2026-10-07 at 11.43.07 (1).jpeg,FAKE,0.000526,REAL,REAL,whatsapp image 2026-10-07 at 11.43.07 (1).jpeg
8,WhatsApp Image 2026-10-07 at 11.43.10 (1).jpeg,REAL,0.000517,REAL,REAL,whatsapp image 2026-10-07 at 11.43.10 (1).jpeg
9,chatgpt-image-30-de-mai.-de-2025-10-30-57.jpg,FAKE,0.000516,REAL,REAL,chatgpt-image-30-de-mai.-de-2025-10-30-57.jpg


In [55]:

import os
import re
import pandas as pd
from pathlib import Path
from difflib import get_close_matches

def normalize_image_name(name):
    name = Path(str(name).strip()).name.casefold()
    name = re.sub(r"^copy of\s*", "", name)
    name = re.sub(r"\s+", " ", name)
    return name

labels_df["nome_norm"] = labels_df["arquivo"].map(normalize_image_name)
pred_df["nome_norm"] = pred_df["arquivo"].map(normalize_image_name)

prediction_names = pred_df["nome_norm"].tolist()
prediction_set = set(prediction_names)

rows = []

for _, row in labels_df.iterrows():
    name = row["nome_norm"]

    if name in prediction_set:
        match = name
        status = "Correspondência exata após normalização"
    else:
        candidates = get_close_matches(
            name, prediction_names, n=3, cutoff=0.6
        )
        match = " | ".join(candidates) if candidates else ""
        status = "Revisar manualmente"

    rows.append({
        "arquivo_rotulo": row["arquivo"],
        "label_real": row["label_real"],
        "candidato_predicao": match,
        "status": status
    })

matching_df = pd.DataFrame(rows)
display(matching_df)


,arquivo_rotulo,label_real,candidato_predicao,status
0,Copy of imagem_falsa (1).jpeg,1,imagem_falsa (1).jpeg,Correspondência exata após normalização
1,Copy of imagem_falsa (1).jpg,1,imagem_falsa (1).jpg,Correspondência exata após normalização
2,Copy of imagem_falsa (2).jpeg,1,imagem_falsa (2).jpeg,Correspondência exata após normalização
3,Copy of imagem_falsa (2).jpg,1,imagem_falsa (2).jpg,Correspondência exata após normalização
4,Copy of imagem_falsa (3).jpeg,1,imagem_falsa (3).jpeg,Correspondência exata após normalização
5,Copy of imagem_falsa (3).jpg,1,imagem_falsa (3).jpg,Correspondência exata após normalização
6,Copy of imagem_falsa (4).jpeg,1,imagem_falsa (4).jpeg,Correspondência exata após normalização
7,Copy of imagem_falsa (4).jpg,1,imagem_falsa (4).jpg,Correspondência exata após normalização
8,Copy of imagem_falsa (5).jpeg,1,imagem_falsa (5).jpeg,Correspondência exata após normalização
9,Copy of imagem_falsa (5).jpg,1,imagem_falsa (5).jpg,Correspondência exata após normalização


In [57]:
import os
import pandas as pd
from pathlib import Path

# Caminhos corretos dos arquivos
LABELS_PATH = (
    "/content/drive/MyDrive/TCC_Final/"
    "validacao_fase3/teste_externo/rotulos.csv"
)

PREDS_PATH = (
    "/content/drive/MyDrive/TCC_Final/"
    "teste_externo/external_predictions.csv"
)

# Carregar os CSVs diretamente, sem reutilizar DataFrames antigos
labels = pd.read_csv(LABELS_PATH)
preds = pd.read_csv(PREDS_PATH)

print("Colunas de rotulos.csv:")
print(labels.columns.tolist())

print("\nColunas de external_predictions.csv:")
print(preds.columns.tolist())

# Validar as colunas necessárias
required_labels = {"arquivo", "label_real"}
required_preds = {"arquivo", "prob_fake"}

missing_labels = required_labels - set(labels.columns)
missing_preds = required_preds - set(preds.columns)

if missing_labels:
    raise ValueError(
        f"Colunas ausentes em rotulos.csv: {sorted(missing_labels)}"
    )

if missing_preds:
    raise ValueError(
        "A coluna prob_fake não foi encontrada em external_predictions.csv. "
        f"Colunas disponíveis: {preds.columns.tolist()}"
    )

# Normalizar nomes, removendo o prefixo 'Copy of '
def normalizar_nome(nome):
    nome = Path(str(nome).strip()).name.casefold()
    if nome.startswith("copy of "):
        nome = nome[len("copy of "):]
    return nome

labels["nome_norm"] = labels["arquivo"].map(normalizar_nome)
preds["nome_norm"] = preds["arquivo"].map(normalizar_nome)

# Verificar duplicatas antes de cruzar
for nome_df, df in [("rotulos.csv", labels), ("predições", preds)]:
    duplicados = df[df["nome_norm"].duplicated(keep=False)]
    if not duplicados.empty:
        print(f"\nNomes normalizados duplicados em {nome_df}:")
        print(duplicados[["arquivo", "nome_norm"]].to_string(index=False))
        raise ValueError(
            f"Há nomes duplicados em {nome_df}; o cruzamento foi interrompido."
        )

# Cruzamento dos rótulos com as probabilidades
audit_df = labels.merge(
    preds[["nome_norm", "prob_fake"]],
    on="nome_norm",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Não calcular métricas se alguma imagem não foi encontrada
nao_encontradas = audit_df[audit_df["_merge"] != "both"]

if not nao_encontradas.empty:
    print("\nImagens sem predição correspondente:")
    print(nao_encontradas[["arquivo", "nome_norm"]].to_string(index=False))
    raise ValueError(
        "O cruzamento não encontrou correspondência para todas as imagens. "
        "Nenhuma métrica foi calculada."
    )

audit_df = audit_df.drop(columns="_merge")

print(f"\nCruzamento concluído: {len(audit_df)} imagens.")
print(audit_df[["arquivo", "label_real", "prob_fake"]].to_string(index=False))

Colunas de rotulos.csv:
['arquivo', 'label_real', 'origem', 'observacao']

Colunas de external_predictions.csv:
['arquivo', 'classe_real', 'prob_fake', 'pred_0.50', 'pred_limiar']

Imagens sem predição correspondente:
                      arquivo             nome_norm
Copy of imagem_falsa (1).jpeg imagem_falsa (1).jpeg
 Copy of imagem_falsa (1).jpg  imagem_falsa (1).jpg
Copy of imagem_falsa (2).jpeg imagem_falsa (2).jpeg
 Copy of imagem_falsa (2).jpg  imagem_falsa (2).jpg
Copy of imagem_falsa (3).jpeg imagem_falsa (3).jpeg
 Copy of imagem_falsa (3).jpg  imagem_falsa (3).jpg
Copy of imagem_falsa (4).jpeg imagem_falsa (4).jpeg
 Copy of imagem_falsa (4).jpg  imagem_falsa (4).jpg
Copy of imagem_falsa (5).jpeg imagem_falsa (5).jpeg
 Copy of imagem_falsa (5).jpg  imagem_falsa (5).jpg
 Copy of imagem_falsa (6).jpg  imagem_falsa (6).jpg
 Copy of imagem_falsa (7).jpg  imagem_falsa (7).jpg
     Copy of imagem_real1.jpg      imagem_real1.jpg
     Copy of imagem_real2.jpg      imagem_real2.jpg
  

ValueError: O cruzamento não encontrou correspondência para todas as imagens. Nenhuma métrica foi calculada.

In [58]:
import pandas as pd

LABELS_PATH = "/content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/rotulos.csv"
PREDS_PATH = "/content/drive/MyDrive/TCC_Final/teste_externo/external_predictions.csv"

labels = pd.read_csv(LABELS_PATH)
preds = pd.read_csv(PREDS_PATH)

print("=== NOMES NO rotulos.csv ===")
for nome in labels["arquivo"]:
    print(repr(nome))

print("\n=== NOMES NO external_predictions.csv ===")
for nome in preds["arquivo"]:
    print(repr(nome))

print("\n=== QUANTIDADES ===")
print("Rótulos:", len(labels))
print("Predições:", len(preds))

=== NOMES NO rotulos.csv ===
'Copy of imagem_falsa (1).jpeg'
'Copy of imagem_falsa (1).jpg'
'Copy of imagem_falsa (2).jpeg'
'Copy of imagem_falsa (2).jpg'
'Copy of imagem_falsa (3).jpeg'
'Copy of imagem_falsa (3).jpg'
'Copy of imagem_falsa (4).jpeg'
'Copy of imagem_falsa (4).jpg'
'Copy of imagem_falsa (5).jpeg'
'Copy of imagem_falsa (5).jpg'
'Copy of imagem_falsa (6).jpg'
'Copy of imagem_falsa (7).jpg'
'Copy of imagem_real1.jpg'
'Copy of imagem_real2.jpg'
'Copy of imagem_real3.jpg'
'Copy of imagem_real4.jpg'
'Copy of imagem_real5.jpg'
'Copy of imagem_real6.jpg'
'Copy of imagem_real7.jpg'
'Copy of imagem_real8.jpg'

=== NOMES NO external_predictions.csv ===
'techtudo-21-.jpg'
'unnamed-37-.jpg'
'gemini-generated-image-6gfhpd6gfhpd6gfh.jpg'
'wooden-bridge-beach-sunset.jpg'
'WhatsApp Image 2026-10-07 at 11.43.08.jpeg'
'WhatsApp Image 2026-10-07 at 11.43.11.jpeg'
'unnamed-29-.jpg'
'WhatsApp Image 2026-10-07 at 11.43.07 (1).jpeg'
'WhatsApp Image 2026-10-07 at 11.43.10 (1).jpeg'
'chatgpt-imag

In [59]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/TCC_Final")
LABELS_PATH = ROOT / "validacao_fase3/teste_externo/rotulos.csv"
PREDS_PATH = ROOT / "teste_externo/external_predictions.csv"
IMAGE_DIR = ROOT / "validacao_fase3/teste_externo"

labels = pd.read_csv(LABELS_PATH)
preds = pd.read_csv(PREDS_PATH)

print("Rótulos:", len(labels), "| Predições existentes:", len(preds))

# Localizar as imagens pelos nomes registrados nos rótulos
image_files = {
    p.name.casefold(): p
    for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
}

def normalize(name):
    return Path(str(name).strip()).name.casefold()

labels["nome_norm"] = labels["arquivo"].map(normalize)
preds["nome_norm"] = preds["arquivo"].map(normalize)

found = labels["nome_norm"].isin(image_files)
print(f"Imagens encontradas pelos nomes dos rótulos: {found.sum()}/{len(labels)}")

if found.all():
    print("\nOs 20 arquivos estão disponíveis. Podemos gerar as predições novamente usando os nomes dos rótulos.")
    print("Arquivos encontrados:")
    for name in labels["arquivo"]:
        print(" -", image_files[normalize(name)])
else:
    print("\nNomes de rótulos sem arquivo correspondente:")
    for name in labels.loc[~found, "arquivo"]:
        print(" -", name)

    print("\nArquivos de imagem disponíveis na pasta:")
    for name in sorted(image_files):
        print(" -", image_files[name].name)

print("\nColunas do CSV de predições:", preds.columns.tolist())

Rótulos: 20 | Predições existentes: 20
Imagens encontradas pelos nomes dos rótulos: 20/20

Os 20 arquivos estão disponíveis. Podemos gerar as predições novamente usando os nomes dos rótulos.
Arquivos encontrados:
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (1).jpeg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (1).jpg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (2).jpeg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (2).jpg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (3).jpeg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (3).jpg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (4).jpeg
 - /content/drive/MyDrive/TCC_Final/validacao_fase3/teste_externo/Copy of imagem_falsa (4).jpg
 - /content/drive/MyDri

In [60]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/TCC_Final")

print("=== CHECKPOINTS DO MODELO ===")
for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".pth", ".pt", ".ckpt", ".safetensors"}:
        print(p)

print("\n=== ARQUIVOS DE CÓDIGO RELACIONADOS À FASE 3 ===")
for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".py", ".ipynb"}:
        if any(term in str(p).lower() for term in ["fase3", "phase3", "inference", "predict", "modelo"]):
            print(p)

=== CHECKPOINTS DO MODELO ===
/content/drive/MyDrive/TCC_Final/modelos/phase1_best_model.pt
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/best.pt
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/last.pt
/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/phase2_final.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/best.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/last.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_multigerador/phase3_final.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_holdout_v2/phase3_final.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_holdout_v2/best.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_holdout_v2/last.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/best.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/last.pt
/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt

=== ARQUIVOS DE CÓDIGO RELACIONADOS À FASE 3

In [61]:
from pathlib import Path
import json

NOTEBOOKS = [
    Path("/content/drive/MyDrive/TCC_Final/Notebooks/fase3 - openfake/fase3.3_compressao.ipynb"),
    Path("/content/drive/MyDrive/TCC_Final/Notebooks/fase3 - openfake/fase3.2_multigerador.ipynb"),
    Path("/content/drive/MyDrive/TCC_Final/Notebooks/fase3 - openfake/fase3.1_multigerador.ipynb"),
]

keywords = [
    "class ", "def forward", "def predict", "def infer",
    "torch.load", "load_state_dict", "transforms.",
    "Normalize(", "IMG_SIZE", "THRESHOLD", "threshold",
    "phase3_final", "best.pt", "state_dict",
    "efficientnet", "vit", "fft", "gate", "checkpoint"
]

for nb_path in NOTEBOOKS:
    if not nb_path.exists():
        print(f"\nNOT FOUND: {nb_path}")
        continue

    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    print("\n" + "=" * 90)
    print("NOTEBOOK:", nb_path.name)

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        if any(k.lower() in source.lower() for k in keywords):
            print(f"\n--- CÉLULA {i} ---")
            print(source[:7000])


NOTEBOOK: fase3.3_compressao.ipynb

--- CÉLULA 3 ---
import os, io, csv, json, math, time, random, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFilter
from sklearn.metrics import confusion_matrix, roc_auc_score

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"

CFG = dict(
    RUN_NAME="fase3_3_compressao_v1",        # pasta do experimento (dentro de .../modelos/)
    PHASE2_CKPT="/content/drive/MyDrive/TCC_Final/modelos/fase2_gate_vit_fft_cnn/phase2_final.pt",

    # --- dados (OpenFake, em streaming) ---
    HF_REPO="ComplexDataLab/OpenFake", HF_CONFIG="core",
    DATA_DIR="/content/data/p3_holdout_v2",
    SHORT_SIDE=512,                       # lado curto máximo salvo (reduz disco e igual

In [54]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
import os

PRED_CSV = os.path.join(EXTERNAL_DIR, "rotulos.csv")

pred_df = pd.read_csv(PRED_CSV)

# Padronizar os nomes dos arquivos para permitir o cruzamento
labels_df["arquivo"] = labels_df["arquivo"].astype(str).apply(
    lambda x: os.path.basename(x)
)
# Identificar a coluna que contém o caminho ou nome da imagem
print("Colunas do CSV:", pred_df.columns.tolist())

path_candidates = ["path", "arquivo", "filename", "file", "image_path", "image"]
path_col = next(
    (col for col in path_candidates if col in pred_df.columns),
    None
)

if path_col is None:
    raise ValueError(
        "Não foi encontrada uma coluna com o caminho/nome da imagem. "
        f"Colunas disponíveis: {pred_df.columns.tolist()}"
    )

pred_df["arquivo"] = pred_df[path_col].astype(str).apply(
    lambda x: os.path.basename(x)
)

print("Coluna utilizada para identificar imagens:", path_col)


# Associar os rótulos reais às probabilidades previstas
audit_df = labels_df.merge(
    pred_df[["arquivo", "prob_fake"]],
    on="arquivo",
    how="inner",
    validate="one_to_one"
)

assert len(audit_df) == len(labels_df), (
    f"Há imagens sem predição ou nomes que não coincidem. "
    f"Rótulos: {len(labels_df)}; correspondências: {len(audit_df)}."
)

THRESHOLD = 0.470

y_true = audit_df["label_real"].astype(int)
y_prob = audit_df["prob_fake"].astype(float)
y_pred = (y_prob >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("========== AUDITORIA DO CONJUNTO EXTERNO ==========")
print("Amostras:", len(audit_df))
print("Limiar:", THRESHOLD)
print("Acurácia:", round(accuracy_score(y_true, y_pred), 4))
print(
    "Acurácia balanceada:",
    round(balanced_accuracy_score(y_true, y_pred), 4)
)
print(
    "Precisão (fake):",
    round(precision_score(y_true, y_pred, zero_division=0), 4)
)
print(
    "Revocação de falsas (recall fake):",
    round(recall_score(y_true, y_pred, zero_division=0), 4)
)
print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("TN, FP, FN, TP:", tn, fp, fn, tp)

# Verificar se as predições salvas correspondem ao limiar aplicado
if "pred_thr" in pred_df.columns:
    pred_check = audit_df[["arquivo"]].merge(
        pred_df[["arquivo", "pred_thr"]],
        on="arquivo",
        how="left",
        validate="one_to_one"
    )

    pred_saved = pred_check["pred_thr"].astype(int).to_numpy()
    pred_calc = y_pred.to_numpy()
    divergences = (pred_saved != pred_calc).sum()

    print("Divergências em pred_thr:", divergences)

    if divergences == 0:
        print("OK: pred_thr corresponde ao limiar aplicado.")
    else:
        print("ATENÇÃO: pred_thr difere das classes recalculadas.")

# Adicionar resultados à tabela de auditoria
audit_df["predicao"] = y_pred
audit_df["acertou"] = audit_df["label_real"].astype(int) == audit_df["predicao"]

display(
    audit_df[
        ["arquivo", "label_real", "prob_fake", "predicao", "acertou", "origem"]
    ].sort_values("prob_fake", ascending=False)
)

Colunas do CSV: ['arquivo', 'label_real', 'origem', 'observacao']
Coluna utilizada para identificar imagens: arquivo


KeyError: "['prob_fake'] not in index"

até aqui


### Checagem de atalhos: reais e fakes diferem em formato ou resolução original?

In [9]:
# Se reais e fakes diferem muito em formato/resolução ORIGINAL, o modelo pode aprender isso em vez de aprender "IA".
# Os arquivos salvos já são todos JPEG com mesma qualidade e lado curto <= 512, mas sinais residuais (dupla compressão,
# tamanho de origem) podem sobrar. Esta tabela serve para você enxergar o risco, não para eliminá-lo.
for split in ("train", "test"):
    jpeg_share = {}
    for lab, nm in ((0, "real"), (1, "fake")):
        sub = [r for r in MANIFEST[split] if r["label"] == lab]
        if not sub:
            continue
        fmts = collections.Counter(r["orig_format"] for r in sub)
        ss = np.array([min(r["orig_w"], r["orig_h"]) for r in sub])
        jpeg_share[nm] = fmts.get("JPEG", 0) / len(sub)
        print(f"{split:5s} {nm:4s} n={len(sub):5d} | formatos originais: {dict(fmts.most_common(4))} | "
              f"lado curto original p10/p50/p90 = {np.percentile(ss, [10, 50, 90]).astype(int).tolist()}")
    if len(jpeg_share) == 2 and abs(jpeg_share["real"] - jpeg_share["fake"]) > 0.5:
        print(f"  >>> ATENÇÃO ({split}): a fração de JPEG original difere muito entre reais e fakes "
              f"({jpeg_share['real']:.0%} x {jpeg_share['fake']:.0%}). Risco de atalho de formato/compressão.")

train real n= 3000 | formatos originais: {'JPEG': 2965, 'PNG': 22, 'WEBP': 13} | lado curto original p10/p50/p90 = [420, 1907, 4160]
train fake n= 4983 | formatos originais: {'JPEG': 2666, 'PNG': 2208, 'WEBP': 109} | lado curto original p10/p50/p90 = [512, 880, 1024]
test  real n= 1000 | formatos originais: {'JPEG': 1000} | lado curto original p10/p50/p90 = [333, 588, 1536]
test  fake n=  947 | formatos originais: {'JPEG': 507, 'PNG': 424, 'WEBP': 16} | lado curto original p10/p50/p90 = [720, 1024, 1440]


### Transforms (em sessão nova, rodar esta célula antes de testar imagens próprias)

In [10]:
NORM_MEAN, NORM_STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)    # normalização para [-1, 1], igual à Fase 2
CROP = CFG["CROP"]


class RandomJPEG:
    # Augmentation: recomprime em JPEG com qualidade aleatória (simula imagens que circularam na internet)
    def __init__(self, p=0.3, quality=(60, 95)):
        self.p, self.quality = p, quality

    def __call__(self, img):
        if random.random() > self.p:
            return img
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(*self.quality))
        buf.seek(0)
        return Image.open(buf).convert("RGB")


class RandomBlur:
    # Augmentation: desfoque gaussiano leve, com probabilidade p
    def __init__(self, p=0.1, radius=(0.3, 1.2)):
        self.p, self.radius = p, radius

    def __call__(self, img):
        if random.random() > self.p:
            return img
        return img.filter(ImageFilter.GaussianBlur(random.uniform(*self.radius)))


class RandomDownUp:
    # Augmentation: reduz e amplia de volta (simula redimensionamentos feitos por apps e redes sociais)
    def __init__(self, p=0.15, scale=(0.5, 0.9)):
        self.p, self.scale = p, scale

    def __call__(self, img):
        if random.random() > self.p:
            return img
        k = random.uniform(*self.scale)
        w, h = img.size
        small = img.resize((max(8, int(w * k)), max(8, int(h * k))), Image.BILINEAR)
        return small.resize((w, h), Image.BICUBIC)


_to_norm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])

# Treino: recorte aleatório 224 de uma imagem de lado curto <= 512 (preserva a textura local, sem Resize destrutivo)
train_tf = transforms.Compose([
    transforms.RandomCrop(CROP, pad_if_needed=True),
    transforms.RandomHorizontalFlip(),
    RandomBlur(0.1),
    RandomDownUp(0.15),
    RandomJPEG(1.0, (55, 95)),
    _to_norm])

# Validação rápida (early stopping): recorte central
eval_tf = transforms.Compose([transforms.CenterCrop(CROP), _to_norm])
# Calibração, teste e uso real: 5 recortes (4 cantos + centro); os logits são promediados
_five = transforms.FiveCrop(CROP)


def multicrop_tf(img):
    return torch.stack([_to_norm(c) for c in _five(img)])    # (5, 3, 224, 224)


# Só para o baseline da Fase 2: o pré-processamento ORIGINAL dela (Resize da imagem inteira para 224x224)
phase2_tf = transforms.Compose([transforms.Resize((CROP, CROP)), _to_norm])

### Splits (estratificados por gerador), amostragem balanceada e loaders

In [11]:
def find_faces_root(base):
    # localiza a pasta com train/real e train/fake dentro de `base` (estrutura do dataset de faces da Fase 2)
    for dirpath, _, _ in os.walk(base):
        if all(os.path.isdir(os.path.join(dirpath, "train", c)) for c in ("real", "fake")):
            return dirpath
    return None


def replay_rows(n_per_class):
    # Opcional: amostra faces da Fase 2 para o modelo não esquecer StyleGAN. Vazio se a pasta não existir.
    base = CFG["REPLAY_DIR"]
    if not base or not os.path.isdir(base):
        return []
    root = find_faces_root(base)
    if root is None:
        return []
    rng, rows = random.Random(CFG["SEED"]), []
    for lab, cls in ((0, "real"), (1, "fake")):
        d = os.path.join(root, "train", cls)
        files = sorted(os.listdir(d))
        rng.shuffle(files)
        rows += [{"path": os.path.join(d, f), "label": lab, "group": f"faces-{cls}"} for f in files[:n_per_class]]
    return rows


def split_by_group(rows, frac, seed):
    # Divide em treino / validação A / validação B mantendo a proporção de CADA grupo (gerador ou fonte).
    # Grupos com menos de 10 imagens vão inteiros para o treino.
    rng = random.Random(seed)
    by_g = collections.defaultdict(list)
    for r in rows:
        by_g[(r["label"], r["group"])].append(r)
    tr, v1, v2 = [], [], []
    for _, lst in sorted(by_g.items()):
        rng.shuffle(lst)
        k = int(len(lst) * frac) if len(lst) >= 10 else 0
        v1 += lst[:k]; v2 += lst[k:2 * k]; tr += lst[2 * k:]
    return tr, v1, v2


def sampler_weights(rows):
    # Peso de cada imagem = 1 / (nº de grupos da classe x tamanho do seu grupo):
    # classes ficam 50/50 e cada gerador/fonte contribui igual, mesmo que tenha menos imagens.
    cnt = collections.Counter((r["label"], r["group"]) for r in rows)
    ngr = collections.Counter(lab for (lab, _) in cnt)
    return [1.0 / (ngr[r["label"]] * cnt[(r["label"], r["group"])]) for r in rows]


class PathDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return self.transform(Image.open(r["path"]).convert("RGB")), r["label"]


train_pool = MANIFEST["train"] + replay_rows(CFG["REPLAY_PER_CLASS"])
tr_rows, ves_rows, vcal_rows = split_by_group(train_pool, CFG["VAL_FRAC"], CFG["SEED"])
test_rows = MANIFEST["test"]

# Garantias: nenhuma imagem em dois conjuntos e nenhum gerador hold-out no treino
paths = [set(r["path"] for r in s) for s in (tr_rows, ves_rows, vcal_rows, test_rows)]
assert all(not (paths[i] & paths[j]) for i in range(4) for j in range(i + 1, 4)), "Há imagens repetidas entre conjuntos."
assert not any(r["group"] in CFG["HOLDOUT"] and r["label"] == 1 for r in train_pool), "Gerador hold-out vazou para o treino."

nw = CFG["NUM_WORKERS"]

print("\n=== CONFERÊNCIA DOS GERADORES HOLD-OUT ===")

n_holdout = sum(
    r["label"] == 1 and r["group"] in CFG["HOLDOUT"]
    for r in test_rows
)

print("Total de imagens fake dos geradores reservados:", n_holdout)

for group in CFG["HOLDOUT"]:
    n_group = sum(
        r["label"] == 1 and r["group"] == group
        for r in test_rows
    )
    print(f"{group}: {n_group} imagens")

assert n_holdout > 0, (
    "Não há imagens dos geradores reservados no teste. "
    "Não continue o experimento."
)

for group in CFG["HOLDOUT"]:
    n_group = sum(
        r["label"] == 1 and r["group"] == group
        for r in test_rows
    )
    assert n_group > 0, (
        f"O gerador {group} não possui imagens no teste."
    )

print("OK: cada gerador reservado possui imagens no teste.")

train_loader = DataLoader(PathDataset(tr_rows, train_tf), batch_size=CFG["BATCH_SIZE"],
                          sampler=WeightedRandomSampler(sampler_weights(tr_rows), num_samples=len(tr_rows), replacement=True),
                          num_workers=nw, pin_memory=True, drop_last=True, persistent_workers=nw > 0)
ves_loader = DataLoader(PathDataset(ves_rows, eval_tf), batch_size=128, shuffle=False, num_workers=nw, pin_memory=True)
vcal_loader = DataLoader(PathDataset(vcal_rows, multicrop_tf), batch_size=16, shuffle=False, num_workers=nw, pin_memory=True)
test_loader = DataLoader(PathDataset(test_rows, multicrop_tf), batch_size=16, shuffle=False, num_workers=nw, pin_memory=True)
base_loader = DataLoader(PathDataset(test_rows, phase2_tf), batch_size=64, shuffle=False, num_workers=nw, pin_memory=True)

print(f"Treino: {len(tr_rows)} | validação (early stopping): {len(ves_rows)} | validação (calibração): {len(vcal_rows)} | teste: {len(test_rows)}")
print(f"Teste: reais={sum(r['label'] == 0 for r in test_rows)} | fakes={sum(r['label'] == 1 for r in test_rows)} "
      f"| fakes de gerador hold-out (não visto)={sum(r['label'] == 1 and r['group'] in CFG['HOLDOUT'] for r in test_rows)}")


=== CONFERÊNCIA DOS GERADORES HOLD-OUT ===
Total de imagens fake dos geradores reservados: 155
gpt-image-2: 92 imagens
nano-banana-pro: 63 imagens
OK: cada gerador reservado possui imagens no teste.
Treino: 6429 | validação (early stopping): 777 | validação (calibração): 777 | teste: 1947
Teste: reais=1000 | fakes=947 | fakes de gerador hold-out (não visto)=155


### Modelo: arquitetura da Fase 2 + pesos do `phase2_final.pt`

In [12]:
# Arquitetura IDÊNTICA à da Fase 2 (nomes de módulos precisam casar com o checkpoint).
# ---------- Componentes da Fase 1 (NÃO alterar: precisam casar com os pesos salvos) ----------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        attn = self.dropout(attn)
        return self.proj((attn @ v).transpose(1, 2).reshape(B, N, C))


class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.act(self.fc1(x)))
        return self.dropout(self.fc2(x))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLPBlock(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class ViTTiny(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
                                     for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = torch.cat([self.cls_token.expand(B, -1, -1), self.patch_embed(x)], dim=1)
        x = self.pos_dropout(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)[:, 0]


class FFTBranch(nn.Module):
    def __init__(self, in_channels=3, out_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, out_dim)

    @staticmethod
    def to_log_magnitude_spectrum(x):
        fft = torch.fft.fftshift(torch.fft.fft2(x, norm="ortho"), dim=(-2, -1))
        return torch.log1p(torch.abs(fft))

    def forward(self, x):
        feat = self.conv(self.to_log_magnitude_spectrum(x))
        return self.fc(self.pool(feat).flatten(1))


# ---------- Novo na Fase 2 ----------
class CNNBranch(nn.Module):
    # EfficientNet-B0 (ImageNet) -> vetor de 1280 dims. Converte a normalização [-1,1] do projeto
    # para a normalização esperada pelos pesos do ImageNet.
    def __init__(self, pretrained=True):
        super().__init__()
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = m.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x * 0.5 + 0.5 - self.mean) / self.std
        return self.pool(self.features(x)).flatten(1)


class Phase2Detector(nn.Module):
    DIMS = {"vit": 192, "fft": 128, "cnn": 1280}

    def __init__(self, branches=("vit", "fft", "cnn"), fusion="gate", proj_dim=256,
                 pretrained_cnn=True, num_classes=2, dropout=0.2):
        super().__init__()
        self.names = [n for n in ("vit", "fft", "cnn") if n in branches]
        self.fusion = fusion
        mods = {}
        if "vit" in self.names: mods["vit"] = ViTTiny(embed_dim=192)
        if "fft" in self.names: mods["fft"] = FFTBranch(out_dim=128)
        if "cnn" in self.names: mods["cnn"] = CNNBranch(pretrained_cnn)
        self.branches = nn.ModuleDict(mods)
        self.proj = nn.ModuleDict({n: nn.Sequential(nn.LayerNorm(self.DIMS[n]), nn.Linear(self.DIMS[n], proj_dim))
                                   for n in self.names})
        k = len(self.names)
        if fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(k * proj_dim, 64), nn.GELU(), nn.Linear(64, k))
            cls_in = proj_dim
        else:
            self.gate = None
            cls_in = k * proj_dim
        self.classifier = nn.Sequential(nn.LayerNorm(cls_in), nn.Linear(cls_in, 128), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x, return_gate=False):
        feats = [self.proj[n](self.branches[n](x)) for n in self.names]
        gate_w = None
        if self.fusion == "gate":
            gate_w = torch.softmax(self.gate(torch.cat(feats, dim=1)), dim=1)          # (B, k)
            fused = (torch.stack(feats, dim=1) * gate_w.unsqueeze(-1)).sum(dim=1)       # (B, D)
        else:
            fused = torch.cat(feats, dim=1)
        logits = self.classifier(fused)
        return (logits, gate_w) if return_gate else logits


# ---------- carrega o checkpoint final da Fase 2 ----------
# weights_only=False porque o arquivo guarda também dicionários (cfg, T, limiar); só abra arquivos seus.
ck2 = torch.load(CFG["PHASE2_CKPT"], map_location="cpu", weights_only=False)
cfg2 = ck2["cfg"]                                           # configuração de arquitetura usada na Fase 2
model = Phase2Detector(branches=cfg2["BRANCHES"], fusion=cfg2["FUSION"], proj_dim=cfg2["PROJ_DIM"],
                       pretrained_cnn=False)                # os pesos vêm do checkpoint
model.load_state_dict(ck2["model_state_dict"])
model = model.to(device)
T2, THR2 = float(ck2["temperature"]), float(ck2["threshold"])   # calibração e limiar da Fase 2 (usados só no baseline)
print(f"Fase 2 carregada | ramos: {model.names} | fusão: {cfg2['FUSION']} | T2={T2:.3f} | limiar2={THR2:.3f} "
      f"| parâmetros: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

Fase 2 carregada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M


### Métricas e utilitários

In [13]:
def binary_metrics(y_true, y_pred):
    # métricas com fake = classe positiva
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn)
    tnr = tn / max(1, tn + fp)
    return {"accuracy": (tp + tn) / len(y_true), "precision": prec, "recall": rec,
            "f1": 2 * prec * rec / max(1e-12, prec + rec), "tnr": tnr, "bal_acc": (rec + tnr) / 2,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


@torch.no_grad()
def collect_center(model, loader):
    # logits com 1 recorte por imagem (loaders de recorte central ou Resize)
    model.eval()
    L, Y = [], []
    for imgs, labels in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg = model(imgs.to(device, non_blocking=True))
        L.append(lg.float().cpu()); Y.append(labels)
    return torch.cat(L), torch.cat(Y)


@torch.no_grad()
def collect_tta(model, loader):
    # logits médios sobre 5 recortes por imagem (loaders com multicrop_tf)
    model.eval()
    L, Y = [], []
    for imgs, labels in loader:                                    # imgs: (B, 5, 3, H, W)
        B = imgs.size(0)
        x = imgs.to(device, non_blocking=True).flatten(0, 1)       # (B*5, 3, H, W)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg = model(x)
        L.append(lg.float().view(B, 5, -1).mean(1).cpu()); Y.append(labels)
    return torch.cat(L), torch.cat(Y)


def evaluate_center(model, loader):
    # validação rápida durante o treino: loss, AUC (independe de limiar) e acurácia balanceada @0.5
    lg, y = collect_center(model, loader)
    p = F.softmax(lg, 1)[:, 1].numpy()
    yn = y.numpy()
    m = binary_metrics(yn, (p >= 0.5).astype(int))
    return {"loss": float(F.cross_entropy(lg, y)), "auc": float(roc_auc_score(yn, p)), "bal_acc": m["bal_acc"], "f1": m["f1"]}


def make_lr_lambda(total, warmup):
    # warmup linear + decaimento cosseno, por época
    def f(e):
        if e < warmup:
            return (e + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (e - warmup) / max(1, total - warmup)))
    return f


def save_atomic(obj, path):
    # grava em arquivo temporário e renomeia: um corte de conexão não corrompe o checkpoint
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def write_history_csv(history, path):
    fields = []
    for h in history:
        for k in h:
            if k not in fields:
                fields.append(k)
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, restval="")
        w.writeheader(); w.writerows(history)


def fit_temperature(logits, labels):
    # temperatura T que minimiza a NLL (calibração por "temperature scaling")
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100, line_search_fn="strong_wolfe")
    ce = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad()
        loss = ce(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def ece_score(p_fake, y, n_bins=15):
    # erro de calibração esperado
    conf = np.maximum(p_fake, 1 - p_fake)
    correct = ((p_fake >= 0.5).astype(int) == y).astype(float)
    edges, ece = np.linspace(0.5, 1.0, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def pick_threshold(p, y, grid=None):
    # limiar que maximiza a acurácia balanceada; empate resolvido pelo mais próximo de 0.5
    grid = np.linspace(0.05, 0.95, 181) if grid is None else grid
    best_t, best_s = 0.5, -1.0
    for t in grid:
        pred = (p >= t).astype(int)
        tpr = ((pred == 1) & (y == 1)).sum() / max(1, (y == 1).sum())
        tnr = ((pred == 0) & (y == 0)).sum() / max(1, (y == 0).sum())
        s = (tpr + tnr) / 2
        if s > best_s + 1e-12 or (abs(s - best_s) <= 1e-12 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_t, best_s = float(t), float(s)
    return best_t, best_s


def wilson_ci(k, n, z=1.96):
    # intervalo de confiança de 95% (Wilson) para uma proporção k/n
    if n == 0:
        return 0.0, 0.0
    p = k / n
    den = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / den
    meia = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, centro - meia), min(1.0, centro + meia)


def bootstrap_auc_ci(y, p, n_boot=1000, seed=0):
    # IC de 95% do AUC por bootstrap (reamostra imagens com reposição)
    rng, vals = np.random.default_rng(seed), []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2:
            vals.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def per_group_table(rows, p_fake, thr):
    # Para cada grupo: n e taxa de acerto. Fake = revocação (fake detectada). Real = TNR (real aceita como real).
    idx = collections.defaultdict(list)
    for i, r in enumerate(rows):
        idx[(r["label"], r["group"])].append(i)
    out = []
    for (lab, g), ii in idx.items():
        pred_fake = int((p_fake[ii] >= thr).sum())
        hit = pred_fake if lab == 1 else len(ii) - pred_fake
        lo, hi = wilson_ci(hit, len(ii))
        out.append({"classe": "fake" if lab == 1 else "real", "grupo": g, "n": len(ii), "acerto": hit / len(ii),
                    "ic_lo": lo, "ic_hi": hi, "holdout": g in CFG["HOLDOUT"]})
    return sorted(out, key=lambda d: (d["classe"], d["acerto"]))


def print_group_table(tab, n_worst=10):
    # imprime: todas as fontes reais, geradores-alvo, hold-out e os n_worst fakes mais difíceis; salvar o CSV completo à parte
    def line(d):
        tag = " (NÃO VISTO NO TREINO)" if d["holdout"] else ""
        return f"  {d['grupo'][:28]:28s} n={d['n']:4d}  acerto={d['acerto']:.2f}  IC95=[{d['ic_lo']:.2f}, {d['ic_hi']:.2f}]{tag}"
    print("REAIS (acerto = TNR):")
    for d in tab:
        if d["classe"] == "real":
            print(line(d))
    fakes = [d for d in tab if d["classe"] == "fake"]
    print("FAKES de interesse (acerto = revocação):")
    for d in fakes:
        if is_priority(d["grupo"]) or d["holdout"]:
            print(line(d))
    rest = [d for d in fakes if not (is_priority(d["grupo"]) or d["holdout"])]
    print(f"FAKES: {n_worst} piores dos demais:")
    for d in rest[:n_worst]:
        print(line(d))
    if fakes:
        print(f"Revocação média POR GERADOR (cada gerador pesa igual): {np.mean([d['acerto'] for d in fakes]):.3f} em {len(fakes)} geradores")


def macro_fake_recall(tab):
    f = [d["acerto"] for d in tab if d["classe"] == "fake"]
    return float(np.mean(f)) if f else float("nan")

### Baseline: a Fase 2 (sem tocar nos pesos) no novo conjunto de teste

In [14]:
# ANTES de treinar: mede a Fase 2 com o pré-processamento e o limiar originais dela (Resize 224 da imagem inteira).
# Isso quantifica a falha que você viu e dá a referência contra a qual a Fase 3 precisa melhorar.
lg_b, y_b_t = collect_center(model, base_loader)
y_b = y_b_t.numpy()
p_b = F.softmax(lg_b / T2, 1)[:, 1].numpy()
auc_b = roc_auc_score(y_b, p_b)
lo_b, hi_b = bootstrap_auc_ci(y_b, p_b)
m_b = binary_metrics(y_b, (p_b >= THR2).astype(int))
tab_b = per_group_table(test_rows, p_b, THR2)
BASE = {"auc": auc_b, "auc_ci": (lo_b, hi_b), "bal_acc": m_b["bal_acc"], "macro_fake_recall": macro_fake_recall(tab_b)}
print(f"FASE 2 no novo teste | AUC={auc_b:.4f} (IC95 {lo_b:.3f}-{hi_b:.3f}) | acurácia balanceada @limiar2={m_b['bal_acc']:.4f} "
      f"| TNR={m_b['tnr']:.3f} | revocação={m_b['recall']:.3f}")
print_group_table(tab_b)

FASE 2 no novo teste | AUC=0.5617 (IC95 0.535-0.588) | acurácia balanceada @limiar2=0.5005 | TNR=1.000 | revocação=0.001
REAIS (acerto = TNR):
  docci                        n= 500  acerto=1.00  IC95=[0.99, 1.00]
  imagenet                     n= 500  acerto=1.00  IC95=[0.99, 1.00]
FAKES de interesse (acerto = revocação):
  gpt-image-1.5                n= 120  acerto=0.00  IC95=[0.00, 0.03]
  gpt-image-2                  n=  92  acerto=0.00  IC95=[0.00, 0.04] (NÃO VISTO NO TREINO)
  nano-banana-pro              n=  63  acerto=0.00  IC95=[0.00, 0.06] (NÃO VISTO NO TREINO)
FAKES: 10 piores dos demais:
  veo-3                        n=  40  acerto=0.00  IC95=[0.00, 0.09]
  z-image-turbo                n=  40  acerto=0.00  IC95=[0.00, 0.09]
  ernie-image-turbo            n=  40  acerto=0.00  IC95=[0.00, 0.09]
  flux.2-klein-9b              n=  40  acerto=0.00  IC95=[0.00, 0.09]
  illustrious                  n=  40  acerto=0.00  IC95=[0.00, 0.09]
  recraft-v3                   n=  40  acer

### Treino (fine-tuning completo, amostragem balanceada por gerador, retomável)

In [15]:
RESUME = False
EPOCHS = CFG["EPOCHS"]
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["LABEL_SMOOTHING"])
scaler = torch.amp.GradScaler("cuda")

for p in model.parameters():                       # fine-tuning COMPLETO: todos os ramos treináveis
    p.requires_grad = True

# um grupo de parâmetros por ramo (vit / fft / cnn) com taxa de aprendizado própria; o resto é "head"
lrs = {"vit": CFG["LR_VIT"], "fft": CFG["LR_FFT"], "cnn": CFG["LR_CNN"], "head": CFG["LR_HEAD"]}
groups = {k: [] for k in lrs}
for n, p in model.named_parameters():
    groups[n.split(".")[1] if n.startswith("branches.") else "head"].append(p)
optimizer = torch.optim.AdamW([{"params": ps, "lr": lrs[k], "name": k} for k, ps in groups.items() if ps],
                              weight_decay=CFG["WEIGHT_DECAY"])
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=make_lr_lambda(EPOCHS, CFG["WARMUP"]))


def train_one_epoch(model, loader, optimizer):
    model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    total, n = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                     # desfaz a escala antes de cortar o gradiente
        torch.nn.utils.clip_grad_norm_(params, CFG["MAX_GRAD_NORM"])
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * imgs.size(0)
        n += imgs.size(0)
    return total / max(1, n)


state = {"epoch": 0, "best_auc": 0.0, "no_improve": 0, "history": []}
if RESUME and os.path.isfile(LAST_CKPT):
    ck = torch.load(LAST_CKPT, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    state.update(epoch=ck["epoch"], best_auc=ck["best_auc"], no_improve=ck["no_improve"], history=ck["history"])
    print(f"Retomando: {state['epoch']}/{EPOCHS} épocas | melhor AUC de validação = {state['best_auc']:.4f}")

stopped = state["no_improve"] >= CFG["PATIENCE"]
while state["epoch"] < EPOCHS and not stopped:
    epoch, t0 = state["epoch"] + 1, time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val = evaluate_center(model, ves_loader)
    lr_head = optimizer.param_groups[-1]["lr"]
    scheduler.step()

    state["epoch"] = epoch
    state["history"].append({"epoch": epoch, "lr_head": lr_head, "train_loss": train_loss,
                             **{f"val_{k}": v for k, v in val.items()}, "sec": time.time() - t0})
    improved = val["auc"] > state["best_auc"]          # critério: AUC de validação (não depende de limiar)
    if improved:
        state["best_auc"], state["no_improve"] = val["auc"], 0
        save_atomic({"model": model.state_dict(), "epoch": epoch, "val": val, "cfg": CFG}, BEST_CKPT)
    else:
        state["no_improve"] += 1
    stopped = state["no_improve"] >= CFG["PATIENCE"]
    save_atomic({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                 "scaler": scaler.state_dict(), "epoch": epoch, "best_auc": state["best_auc"],
                 "no_improve": state["no_improve"], "history": state["history"], "cfg": CFG}, LAST_CKPT)
    write_history_csv(state["history"], HISTORY_CSV)
    print(f"[E{epoch:02d}/{EPOCHS}] lr={lr_head:.1e} | train_loss={train_loss:.4f} | val_loss={val['loss']:.4f} | "
          f"AUC={val['auc']:.4f} | bal_acc={val['bal_acc']:.4f} | {time.time() - t0:.0f}s" + ("  -> melhor" if improved else ""))
    if stopped:
        print(f"Early stopping: sem melhora do AUC em {CFG['PATIENCE']} épocas.")

best = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(best["model"])
print(f"\nMelhor modelo: época {best['epoch']} (AUC de validação = {best['val']['auc']:.4f}) carregado.")

[E01/20] lr=1.0e-04 | train_loss=0.7616 | val_loss=0.6191 | AUC=0.7115 | bal_acc=0.6600 | 54s  -> melhor
[E02/20] lr=1.0e-04 | train_loss=0.6020 | val_loss=0.5491 | AUC=0.7900 | bal_acc=0.7030 | 35s  -> melhor
[E03/20] lr=9.9e-05 | train_loss=0.5605 | val_loss=0.5216 | AUC=0.8111 | bal_acc=0.7114 | 37s  -> melhor
[E04/20] lr=9.7e-05 | train_loss=0.5276 | val_loss=0.4929 | AUC=0.8347 | bal_acc=0.7478 | 36s  -> melhor
[E05/20] lr=9.4e-05 | train_loss=0.5130 | val_loss=0.4882 | AUC=0.8347 | bal_acc=0.7657 | 36s
[E06/20] lr=8.9e-05 | train_loss=0.5041 | val_loss=0.4697 | AUC=0.8507 | bal_acc=0.7721 | 37s  -> melhor
[E07/20] lr=8.4e-05 | train_loss=0.4860 | val_loss=0.4577 | AUC=0.8592 | bal_acc=0.7797 | 36s  -> melhor
[E08/20] lr=7.7e-05 | train_loss=0.4743 | val_loss=0.4523 | AUC=0.8619 | bal_acc=0.7863 | 37s  -> melhor
[E09/20] lr=7.0e-05 | train_loss=0.4643 | val_loss=0.4440 | AUC=0.8668 | bal_acc=0.7934 | 36s  -> melhor
[E10/20] lr=6.2e-05 | train_loss=0.4549 | val_loss=0.4437 | AUC=0.

### Calibração (T e limiar na validação) e teste por gerador, comparado com a Fase 2

In [16]:
# 1) Calibração e limiar: só com a fatia de validação B (nunca vista no treino nem no early stopping), em modo 5 recortes
lg_cal, y_cal_t = collect_tta(model, vcal_loader)
y_cal = y_cal_t.numpy()
T = fit_temperature(lg_cal, y_cal_t)
p_cal = F.softmax(lg_cal / T, 1)[:, 1].numpy()
thr, bal_cal = pick_threshold(p_cal, y_cal)
print(f"T = {T:.3f} | ECE {ece_score(F.softmax(lg_cal, 1)[:, 1].numpy(), y_cal):.4f} -> {ece_score(p_cal, y_cal):.4f} "
      f"| limiar = {thr:.3f} (acurácia balanceada na validação = {bal_cal:.4f})")

# 2) Teste (uma única vez), em modo 5 recortes
lg_te, y_te_t = collect_tta(model, test_loader)
y_te = y_te_t.numpy()
p_te = F.softmax(lg_te / T, 1)[:, 1].numpy()
auc = roc_auc_score(y_te, p_te)
lo, hi = bootstrap_auc_ci(y_te, p_te)
r05, rthr = binary_metrics(y_te, (p_te >= 0.5).astype(int)), binary_metrics(y_te, (p_te >= thr).astype(int))
tab = per_group_table(test_rows, p_te, thr)

print(f"\nFASE 3 no teste | AUC={auc:.4f} (IC95 {lo:.3f}-{hi:.3f})")
print(f"{'':12s}{'@0.5':>10s}{'@limiar':>10s}")
for k in ("accuracy", "precision", "recall", "f1", "tnr", "bal_acc"):
    print(f"{k:12s}{r05[k]:10.4f}{rthr[k]:10.4f}")
print(f"confusão @limiar: TN={rthr['tn']} FP={rthr['fp']} FN={rthr['fn']} TP={rthr['tp']}\n")
print_group_table(tab)

print("\n=== ANTES (Fase 2) x DEPOIS (Fase 3), mesmo conjunto de teste ===")
print(f"AUC:                         {BASE['auc']:.4f}  ->  {auc:.4f}")
print(f"Acurácia balanceada:         {BASE['bal_acc']:.4f}  ->  {rthr['bal_acc']:.4f}")
print(f"Revocação média por gerador: {BASE['macro_fake_recall']:.4f}  ->  {macro_fake_recall(tab):.4f}")
ho = [d["acerto"] for d in tab if d["classe"] == "fake" and d["holdout"]]
vistos = [d["acerto"] for d in tab if d["classe"] == "fake" and not d["holdout"]]
if ho and vistos:
    print(f"Geradores VISTOS no treino: {np.mean(vistos):.3f} | NÃO VISTOS (hold-out): {np.mean(ho):.3f}  <- esta diferença é a medida de generalização")

# 3) Artefatos
with open(os.path.join(RUN_DIR, "per_group.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(tab[0].keys()))
    w.writeheader(); w.writerows(tab)
with open(os.path.join(RUN_DIR, "test_predictions.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["path", "label", "group", "prob_fake", "pred_thr"])
    for r, pp in zip(test_rows, p_te):
        w.writerow([r["path"], r["label"], r["group"], f"{pp:.6f}", int(pp >= thr)])
with open(os.path.join(RUN_DIR, "final_metrics.json"), "w") as f:
    json.dump({"temperature": T, "threshold": thr, "auc": auc, "auc_ci95": [lo, hi], "test_0.5": r05, "test_thr": rthr,
               "baseline_phase2": BASE, "best_epoch": best["epoch"], "cfg": CFG}, f, indent=2, default=str)
torch.save({"model_state_dict": model.state_dict(), "temperature": T, "threshold": thr,
            "cfg": cfg2,                                    # arquitetura (mesmas chaves do phase2_final.pt)
            "cfg_phase3": CFG,
            "preproc": "lado curto<=512, JPEG q92, 5 recortes 224 (4 cantos+centro), logits promediados"},
           os.path.join(RUN_DIR, "phase3_final.pt"))
print(f"\nSalvo em {RUN_DIR}: phase3_final.pt, final_metrics.json, per_group.csv, test_predictions.csv, history.csv")

T = 0.650 | ECE 0.0767 -> 0.0404 | limiar = 0.470 (acurácia balanceada na validação = 0.8458)

FASE 3 no teste | AUC=0.8041 (IC95 0.785-0.822)
                  @0.5   @limiar
accuracy        0.7072    0.7114
precision       0.7869    0.7762
recall          0.5459    0.5713
f1              0.6446    0.6582
tnr             0.8600    0.8440
bal_acc         0.7030    0.7076
confusão @limiar: TN=844 FP=156 FN=406 TP=541

REAIS (acerto = TNR):
  docci                        n= 500  acerto=0.76  IC95=[0.72, 0.79]
  imagenet                     n= 500  acerto=0.93  IC95=[0.90, 0.95]
FAKES de interesse (acerto = revocação):
  gpt-image-1.5                n= 120  acerto=0.49  IC95=[0.40, 0.58]
  gpt-image-2                  n=  92  acerto=0.57  IC95=[0.46, 0.66] (NÃO VISTO NO TREINO)
  nano-banana-pro              n=  63  acerto=0.62  IC95=[0.50, 0.73] (NÃO VISTO NO TREINO)
FAKES: 10 piores dos demais:
  flux.2-klein-9b              n=  40  acerto=0.23  IC95=[0.12, 0.38]
  frames-23-1-25       

### Suas imagens (Gemini, ChatGPT etc.)
Em **sessão nova** rode só: configuração, Drive, Transforms, Modelo, Métricas e utilitários e esta célula (ela carrega o `phase3_final.pt` sozinha; não baixa dados nem treina). O pré-processamento é o mesmo do treino (lado curto ≤ 512, JPEG q92, 5 recortes).

In [18]:
# Carrega o modelo final da Fase 3 se ele ainda não está na memória (sessão nova)
if "T" not in globals() or "thr" not in globals():
    ck3 = torch.load(os.path.join(RUN_DIR, "phase3_final.pt"), map_location="cpu", weights_only=False)
    cfg3 = ck3["cfg"]
    model = Phase2Detector(branches=cfg3["BRANCHES"], fusion=cfg3["FUSION"], proj_dim=cfg3["PROJ_DIM"], pretrained_cnn=False)
    model.load_state_dict(ck3["model_state_dict"])
    model = model.to(device)
    T, thr = float(ck3["temperature"]), float(ck3["threshold"])
model.eval()
print(f"Modelo pronto | T={T:.3f} | limiar={thr:.3f}")


def prep_like_training(im):
    # Replica o que os dados de treino sofreram: RGB, lado curto <= 512 e uma passada por JPEG q92
    im = im.convert("RGB")
    s = min(im.size)
    if s > CFG["SHORT_SIDE"]:
        k = CFG["SHORT_SIDE"] / s
        im = im.resize((round(im.width * k), round(im.height * k)), Image.LANCZOS)
    if min(im.size) < CROP:                       # menor que um recorte: amplia (resultado menos confiável)
        k = CROP / min(im.size)
        im = im.resize((math.ceil(im.width * k), math.ceil(im.height * k)), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=CFG["JPEG_QUALITY"])
    buf.seek(0)
    return Image.open(buf).convert("RGB")


@torch.no_grad()
def predict_pils(pils, chunk=8):
    # devolve P(fake) calibrada e os pesos do gate (média dos 5 recortes) para uma lista de imagens PIL
    probs, gates = [], []
    for i in range(0, len(pils), chunk):
        x = torch.stack([multicrop_tf(prep_like_training(im)) for im in pils[i:i + chunk]]).to(device)   # (B,5,3,H,W)
        B = x.size(0)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg, g = model(x.flatten(0, 1), return_gate=True)
        probs.append(F.softmax(lg.float().view(B, 5, -1).mean(1) / T, 1)[:, 1].cpu())
        if g is not None:
            gates.append(g.float().view(B, 5, -1).mean(1).cpu())
    return torch.cat(probs).numpy(), (torch.cat(gates).numpy() if gates else None)


# ---------- A) Teste rápido: upload ou pasta do Drive ----------
import matplotlib.pyplot as plt
PASTA_DRIVE = "/content/drive/MyDrive/TCC_Final/minhas_imagens"
EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
if PASTA_DRIVE:
    nomes = sorted(f for f in os.listdir(PASTA_DRIVE) if f.lower().endswith(EXTS))
    imagens = [(n, Image.open(os.path.join(PASTA_DRIVE, n))) for n in nomes]
else:
    from google.colab import files as colab_files
    up = colab_files.upload()
    imagens = [(n, Image.open(io.BytesIO(b))) for n, b in up.items() if n.lower().endswith(EXTS)]

if imagens:
    pf, gw = predict_pils([im for _, im in imagens])
    print(f"{'arquivo':40s}{'P(fake)':>9s}  decisão" + ("".join(f"{'gate_' + n:>10s}" for n in model.names) if gw is not None else ""))
    for i, (nome, im) in enumerate(imagens):
        aviso = "  (lado curto < 256: fora do que o modelo viu)" if min(im.size) < CFG["MIN_SIDE"] else ""
        print(f"{nome[:39]:40s}{pf[i]:9.3f}  {'FAKE' if pf[i] >= thr else 'REAL':5s}"
              + ("".join(f"{v:10.2f}" for v in gw[i]) if gw is not None else "") + aviso)
    cols = 4
    rows_n = max(1, math.ceil(len(imagens) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 4.3 * rows_n), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (nome, im), pp in zip(axes.ravel(), imagens, pf):
        ax.imshow(im.convert("RGB"))
        ax.set_title(f"{nome[:28]}\nP(fake)={pp:.2f} -> {'FAKE' if pp >= thr else 'REAL'}", fontsize=9)
    plt.tight_layout(); plt.show()

# ---------- B) Avaliação externa rotulada (opcional): pasta com real/ e fake/ ----------
EXT_DIR = "/content/drive/MyDrive/TCC_Final/teste_externo"
if EXT_DIR:
    ys, ps = [], []
    for lab, cls in ((0, "real"), (1, "fake")):
        d = os.path.join(EXT_DIR, cls)
        fs = [os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(EXTS)]
        for i in range(0, len(fs), 64):                         # em blocos, para não encher a memória
            pils = [Image.open(p) for p in fs[i:i + 64]]
            p_, _ = predict_pils(pils)
            ps += p_.tolist(); ys += [lab] * len(pils)
    ys, ps = np.array(ys), np.array(ps)
    m = binary_metrics(ys, (ps >= thr).astype(int))
    print(f"Externo: {len(ys)} imagens (reais={(ys == 0).sum()}, fakes={(ys == 1).sum()}) | "
          f"acurácia balanceada={m['bal_acc']:.3f} | TNR={m['tnr']:.3f} | revocação={m['recall']:.3f}")
    if len(set(ys)) == 2:
        lo_e, hi_e = bootstrap_auc_ci(ys, ps)
        print(f"AUC={roc_auc_score(ys, ps):.4f} (IC95 {lo_e:.3f}-{hi_e:.3f})  <- com poucas imagens o intervalo será enorme")

Output hidden; open in https://colab.research.google.com to view.